In [1]:
from __future__ import annotations

import importlib
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """Find the project root from either the notebook directory or project root."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "restoration_eval").is_dir() and (
            candidate / "tools" / "build_project_inventory.py"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Could not find project root. Run this notebook from the thesis project or notebooks/ directory."
    )


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

NOTEBOOK_ID = "22_stable_diffusion_classical_metrics"
NOTEBOOK_TITLE = "Stable Diffusion Classical Metrics"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / NOTEBOOK_ID

OUTPUT_DIRS = {
    "inventory": OUTPUT_ROOT / "inventory",
    "validation": OUTPUT_ROOT / "validation",
    "manifests": OUTPUT_ROOT / "manifests",
    "metrics": OUTPUT_ROOT / "metrics",
    "analysis": OUTPUT_ROOT / "analysis",
    "figures": OUTPUT_ROOT / "figures",
    "reports": OUTPUT_ROOT / "reports",
    "tables": OUTPUT_ROOT / "tables",
}

for directory in OUTPUT_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

INVENTORY_CSV_PATH = PROJECT_ROOT / "outputs" / "inventory" / "project_file_inventory.csv"
BATCH0_INVENTORY_SNAPSHOT_PATH = OUTPUT_DIRS["inventory"] / "batch0_project_inventory_snapshot.csv"
BATCH0_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch0_validation.csv"
STAGE_MANIFEST_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_classical_metrics_stage_manifest.json"
ARTIFACT_INDEX_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_classical_metrics_artifact_index.csv"
HANDOFF_MANIFEST_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_classical_metrics_handoff_manifest.json"

UPSTREAM_ROOT = PROJECT_ROOT / "outputs" / "21_stable_diffusion_restoration"

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook output root: {OUTPUT_ROOT}")
print(f"Python: {sys.version.split()[0]} | platform: {platform.platform()}")

Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Notebook output root: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\22_stable_diffusion_classical_metrics
Python: 3.12.6 | platform: Windows-11-10.0.26200-SP0


In [2]:
INVENTORY_SCRIPT_PATH = PROJECT_ROOT / "tools" / "build_project_inventory.py"

if not INVENTORY_SCRIPT_PATH.is_file():
    raise FileNotFoundError(f"Inventory script not found: {INVENTORY_SCRIPT_PATH}")

inventory_command = [
    sys.executable,
    str(INVENTORY_SCRIPT_PATH),
    "--root",
    str(PROJECT_ROOT),
    "--out-dir",
    str(PROJECT_ROOT / "outputs" / "inventory"),
]

print("Refreshing project inventory...")
inventory_result = subprocess.run(
    inventory_command,
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(inventory_result.stdout.strip())

if not INVENTORY_CSV_PATH.is_file():
    raise FileNotFoundError(f"Inventory CSV was not created: {INVENTORY_CSV_PATH}")

project_inventory_df = pd.read_csv(INVENTORY_CSV_PATH)
project_inventory_df.to_csv(BATCH0_INVENTORY_SNAPSHOT_PATH, index=False)

print(f"Inventory rows: {len(project_inventory_df):,}")
print(f"Saved Batch 0 inventory snapshot: {BATCH0_INVENTORY_SNAPSHOT_PATH.relative_to(PROJECT_ROOT)}")

display(project_inventory_df.head(10))

Refreshing project inventory...
Saved inventory: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\inventory\project_file_inventory.csv
Saved summary:   D:\Masters\FH\Thesis\painting-restoration-eval\outputs\inventory\project_file_inventory_summary.csv
Files indexed:   4123
Inventory rows: 4,123
Saved Batch 0 inventory snapshot: outputs\22_stable_diffusion_classical_metrics\inventory\batch0_project_inventory_snapshot.csv


,relative_path,file_name,parent_dir,extension,file_kind,size_bytes,last_modified_iso,depth,csv_row_count,csv_column_count,csv_columns,csv_error,image_width,image_height,image_mode,image_error,sha256_first_1mb
0,.gitattributes,.gitattributes,NaN,NaN,other,60,2026-07-06T17:52:52.879628+00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,439ad27a2f48aacaaf48efa19bc2f8b8fcb5b402d23454...
1,.gitignore,.gitignore,NaN,NaN,other,643,2026-07-06T17:53:38.894066+00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,d4ef631ec54b12485b0f98e910cc14d435433c33f26489...
2,config/experiment_50_config.yaml,experiment_50_config.yaml,config,.yaml,other,7111,2026-07-23T12:34:35.693189+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7af22deb75434907302e3e460b4c28a73f25533f3274e4...
3,config/pilot_config.yaml,pilot_config.yaml,config,.yaml,other,677,2026-06-29T14:26:59.322671+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9c81e4ee3aec9dc1b86aa6bc28f5ba442b694810d83eed...
4,data/model_audit/model_candidates.csv,model_candidates.csv,data/model_audit,.csv,csv,5383,2026-07-03T09:14:19.349960+00:00,2,5.0,17.0,model_name | model_family | open_or_closed | d...,NaN,NaN,NaN,NaN,NaN,4ec7ee8d89e5a9d122ea5e9dd37efa6346a6f686263d66...
5,data/processed/clean/p001_clean.png,p001_clean.png,data/processed/clean,.png,clean_image,723881,2026-07-24T16:38:17.470158+00:00,3,NaN,NaN,NaN,NaN,768.0,768.0,RGB,NaN,b5852f0337c8342a1a70268f9e9f682534177581d23871...
6,data/processed/clean/p002_clean.png,p002_clean.png,data/processed/clean,.png,clean_image,339472,2026-07-24T16:38:18.292719+00:00,3,NaN,NaN,NaN,NaN,768.0,768.0,RGB,NaN,3f173a947e4c2be2ebdc267f0c852f490981bc81da6c4e...
7,data/processed/clean/p003_clean.png,p003_clean.png,data/processed/clean,.png,clean_image,648764,2026-07-24T16:38:19.476394+00:00,3,NaN,NaN,NaN,NaN,768.0,768.0,RGB,NaN,1e2ef6b59b4383d7d44ce2f9bb016c126633d2c3e4f39b...
8,data/processed/clean/p004_clean.png,p004_clean.png,data/processed/clean,.png,clean_image,709746,2026-07-24T16:38:20.522805+00:00,3,NaN,NaN,NaN,NaN,768.0,768.0,RGB,NaN,ae8cfa01d84bd2e8c2f41cfdfe1cb25b6f2abf34301351...
9,data/processed/clean/p005_clean.png,p005_clean.png,data/processed/clean,.png,clean_image,745221,2026-07-24T16:38:21.522086+00:00,3,NaN,NaN,NaN,NaN,768.0,768.0,RGB,NaN,da4c53324f40a8a662edb554ee6215db2a53bcb94fcdb7...


In [3]:
import restoration_eval.metrics_classical as metrics_classical

metrics_classical = importlib.reload(metrics_classical)

EXPECTED_METRIC_VERSION = "2.1.0"
EXPECTED_BASE_REGIONS = ("full_image", "content_region")
EXPECTED_MASKED_CASE_REGIONS = (
    "masked_region",
    "mask_bbox_crop",
    "boundary_region",
    "outside_mask_region",
)
EXPECTED_ALL_REGIONS = EXPECTED_BASE_REGIONS + EXPECTED_MASKED_CASE_REGIONS

REQUIRED_HELPER_SYMBOLS = [
    "METRIC_VERSION",
    "BASE_REGIONS",
    "MASKED_CASE_REGIONS",
    "ALL_EVALUATION_REGIONS",
    "CLASSICAL_METRIC_COLUMNS",
    "CANDIDATE_METADATA_COLUMNS",
    "compute_classical_metrics_for_restorations",
    "expected_metric_rows_from_metadata",
    "expected_region_counts_from_metadata",
    "validate_classical_metrics",
    "summarize_classical_metrics",
]

REQUIRED_CANDIDATE_METADATA_COLUMNS = [
    "restoration_case_id",
    "candidate_id",
    "source_case_key",
    "prompt_policy_id",
    "prompt_variant_id",
    "prompt_template_name",
    "candidate_seed",
    "effective_candidate_seed",
    "restoration_generator_version",
    "restored_path",
    "damaged_path",
    "mask_path",
    "clean_path",
]

helper_api_rows = []
for symbol_name in REQUIRED_HELPER_SYMBOLS:
    helper_api_rows.append(
        {
            "check_name": f"helper_symbol__{symbol_name}",
            "observed": str(hasattr(metrics_classical, symbol_name)),
            "expected": "True",
            "passed": bool(hasattr(metrics_classical, symbol_name)),
            "failure_message": "" if hasattr(metrics_classical, symbol_name) else f"Missing helper symbol: {symbol_name}",
        }
    )

helper_metadata_columns = tuple(getattr(metrics_classical, "CANDIDATE_METADATA_COLUMNS", ()))
missing_metadata_columns = [
    column for column in REQUIRED_CANDIDATE_METADATA_COLUMNS if column not in helper_metadata_columns
]

helper_api_rows.extend(
    [
        {
            "check_name": "helper_metric_version",
            "observed": getattr(metrics_classical, "METRIC_VERSION", ""),
            "expected": EXPECTED_METRIC_VERSION,
            "passed": getattr(metrics_classical, "METRIC_VERSION", "") == EXPECTED_METRIC_VERSION,
            "failure_message": ""
            if getattr(metrics_classical, "METRIC_VERSION", "") == EXPECTED_METRIC_VERSION
            else "Wrong metrics_classical helper version. Replace src/restoration_eval/metrics_classical.py with v2.1.0.",
        },
        {
            "check_name": "helper_base_regions_match_policy",
            "observed": str(tuple(getattr(metrics_classical, "BASE_REGIONS", ()))),
            "expected": str(EXPECTED_BASE_REGIONS),
            "passed": tuple(getattr(metrics_classical, "BASE_REGIONS", ())) == EXPECTED_BASE_REGIONS,
            "failure_message": "",
        },
        {
            "check_name": "helper_masked_regions_match_policy",
            "observed": str(tuple(getattr(metrics_classical, "MASKED_CASE_REGIONS", ()))),
            "expected": str(EXPECTED_MASKED_CASE_REGIONS),
            "passed": tuple(getattr(metrics_classical, "MASKED_CASE_REGIONS", ())) == EXPECTED_MASKED_CASE_REGIONS,
            "failure_message": "",
        },
        {
            "check_name": "helper_all_regions_match_policy",
            "observed": str(tuple(getattr(metrics_classical, "ALL_EVALUATION_REGIONS", ()))),
            "expected": str(EXPECTED_ALL_REGIONS),
            "passed": tuple(getattr(metrics_classical, "ALL_EVALUATION_REGIONS", ())) == EXPECTED_ALL_REGIONS,
            "failure_message": "",
        },
        {
            "check_name": "helper_candidate_metadata_passthrough",
            "observed": str(missing_metadata_columns),
            "expected": "[]",
            "passed": len(missing_metadata_columns) == 0,
            "failure_message": ""
            if not missing_metadata_columns
            else f"Missing candidate metadata passthrough columns: {missing_metadata_columns}",
        },
    ]
)

helper_api_check_df = pd.DataFrame(helper_api_rows)
display(helper_api_check_df)

if not helper_api_check_df["passed"].all():
    display(helper_api_check_df.loc[~helper_api_check_df["passed"], ["check_name", "failure_message"]])
    raise RuntimeError("Batch 0 helper API check failed. Fix the helper before continuing.")

print(f"metrics_classical helper OK: version {metrics_classical.METRIC_VERSION}")

,check_name,observed,expected,passed,failure_message
0,helper_symbol__METRIC_VERSION,True,True,True,
1,helper_symbol__BASE_REGIONS,True,True,True,
2,helper_symbol__MASKED_CASE_REGIONS,True,True,True,
3,helper_symbol__ALL_EVALUATION_REGIONS,True,True,True,
4,helper_symbol__CLASSICAL_METRIC_COLUMNS,True,True,True,
5,helper_symbol__CANDIDATE_METADATA_COLUMNS,True,True,True,
6,helper_symbol__compute_classical_metrics_for_r...,True,True,True,
7,helper_symbol__expected_metric_rows_from_metadata,True,True,True,
8,helper_symbol__expected_region_counts_from_met...,True,True,True,
9,helper_symbol__validate_classical_metrics,True,True,True,


metrics_classical helper OK: version 2.1.0


In [4]:
REQUIRED_UPSTREAM_ARTIFACTS = [
    {
        "artifact_key": "notebook21_handoff_manifest",
        "artifact_type": "json",
        "path": UPSTREAM_ROOT / "manifests" / "stable_diffusion_handoff_manifest.json",
        "required": True,
        "notes": "Notebook 21 final handoff manifest.",
    },
    {
        "artifact_key": "notebook21_artifact_index",
        "artifact_type": "csv",
        "path": UPSTREAM_ROOT / "manifests" / "stable_diffusion_stage_artifact_index.csv",
        "required": True,
        "notes": "Notebook 21 artifact index.",
    },
    {
        "artifact_key": "sd_candidate_manifest",
        "artifact_type": "csv",
        "path": UPSTREAM_ROOT / "stable_diffusion_candidate_manifest.csv",
        "required": True,
        "notes": "Candidate-level input table, expected 945 rows after Notebook 21.",
    },
    {
        "artifact_key": "sd_restoration_audit",
        "artifact_type": "csv",
        "path": UPSTREAM_ROOT / "stable_diffusion_restoration_audit.csv",
        "required": True,
        "notes": "Generation audit, expected all status ok after final Notebook 21 run.",
    },
    {
        "artifact_key": "sd_restoration_validation",
        "artifact_type": "csv",
        "path": UPSTREAM_ROOT / "validation" / "stable_diffusion_restoration_validation.csv",
        "required": True,
        "notes": "Notebook 21 output validation table.",
    },
    {
        "artifact_key": "metadata_processed_clean",
        "artifact_type": "csv",
        "path": PROJECT_ROOT / "data" / "processed" / "metadata" / "metadata_processed_clean.csv",
        "required": True,
        "notes": "Fallback source for clean paths and painting metadata when needed.",
    },
]

PLANNED_OUTPUT_ARTIFACTS = [
    {
        "artifact_key": "batch0_project_inventory_snapshot",
        "batch": "batch0",
        "artifact_type": "csv",
        "path": BATCH0_INVENTORY_SNAPSHOT_PATH,
        "required": True,
        "notes": "Inventory snapshot captured at notebook start.",
    },
    {
        "artifact_key": "batch0_validation",
        "batch": "batch0",
        "artifact_type": "csv",
        "path": BATCH0_VALIDATION_PATH,
        "required": True,
        "notes": "Batch 0 validation checks.",
    },
    {
        "artifact_key": "stage_manifest",
        "batch": "batch0_to_final",
        "artifact_type": "json",
        "path": STAGE_MANIFEST_PATH,
        "required": True,
        "notes": "Notebook-wide stage manifest updated during the run.",
    },
    {
        "artifact_key": "metric_input_cases",
        "batch": "batch1",
        "artifact_type": "csv",
        "path": OUTPUT_DIRS["tables"] / "stable_diffusion_classical_metric_input_cases.csv",
        "required": True,
        "notes": "Standardized candidate-level metric input table.",
    },
    {
        "artifact_key": "classical_metrics",
        "batch": "batch3",
        "artifact_type": "csv",
        "path": OUTPUT_DIRS["metrics"] / "stable_diffusion_classical_metrics.csv",
        "required": True,
        "notes": "Main candidate-level classical metrics table.",
    },
    {
        "artifact_key": "classical_metrics_summary",
        "batch": "batch4",
        "artifact_type": "csv",
        "path": OUTPUT_DIRS["analysis"] / "stable_diffusion_classical_metrics_summary.csv",
        "required": True,
        "notes": "Single consolidated summary table with summary_scope column.",
    },
    {
        "artifact_key": "selected_cases",
        "batch": "batch4",
        "artifact_type": "csv",
        "path": OUTPUT_DIRS["analysis"] / "stable_diffusion_classical_metric_selected_cases.csv",
        "required": True,
        "notes": "Representative cases for visual/diagnostic review.",
    },
    {
        "artifact_key": "final_validation",
        "batch": "batch5",
        "artifact_type": "csv",
        "path": OUTPUT_DIRS["validation"] / "stable_diffusion_classical_metrics_final_validation.csv",
        "required": True,
        "notes": "End-of-notebook validation checks.",
    },
    {
        "artifact_key": "artifact_index",
        "batch": "batch5",
        "artifact_type": "csv",
        "path": ARTIFACT_INDEX_PATH,
        "required": True,
        "notes": "Final index of important notebook outputs.",
    },
    {
        "artifact_key": "handoff_manifest",
        "batch": "batch5",
        "artifact_type": "json",
        "path": HANDOFF_MANIFEST_PATH,
        "required": True,
        "notes": "Final manifest for downstream notebooks.",
    },
]

upstream_artifact_df = pd.DataFrame(
    [
        {
            **{k: v for k, v in artifact.items() if k != "path"},
            "path": str(artifact["path"].relative_to(PROJECT_ROOT)),
            "exists": artifact["path"].is_file(),
            "size_bytes": artifact["path"].stat().st_size if artifact["path"].is_file() else 0,
        }
        for artifact in REQUIRED_UPSTREAM_ARTIFACTS
    ]
)

planned_output_artifact_df = pd.DataFrame(
    [
        {
            **{k: v for k, v in artifact.items() if k != "path"},
            "path": str(artifact["path"].relative_to(PROJECT_ROOT)),
            "exists": artifact["path"].is_file(),
            "size_bytes": artifact["path"].stat().st_size if artifact["path"].is_file() else 0,
        }
        for artifact in PLANNED_OUTPUT_ARTIFACTS
    ]
)

print("Required upstream artifacts:")
display(upstream_artifact_df)

print("Planned important outputs:")
display(planned_output_artifact_df)

Required upstream artifacts:


,artifact_key,artifact_type,required,notes,path,exists,size_bytes
0,notebook21_handoff_manifest,json,True,Notebook 21 final handoff manifest.,outputs\21_stable_diffusion_restoration\manife...,True,21956
1,notebook21_artifact_index,csv,True,Notebook 21 artifact index.,outputs\21_stable_diffusion_restoration\manife...,True,16278
2,sd_candidate_manifest,csv,True,"Candidate-level input table, expected 945 rows...",outputs\21_stable_diffusion_restoration\stable...,True,2904218
3,sd_restoration_audit,csv,True,"Generation audit, expected all status ok after...",outputs\21_stable_diffusion_restoration\stable...,True,3512362
4,sd_restoration_validation,csv,True,Notebook 21 output validation table.,outputs\21_stable_diffusion_restoration\valida...,True,438181
5,metadata_processed_clean,csv,True,Fallback source for clean paths and painting m...,data\processed\metadata\metadata_processed_cle...,True,30443


Planned important outputs:


,artifact_key,batch,artifact_type,required,notes,path,exists,size_bytes
0,batch0_project_inventory_snapshot,batch0,csv,True,Inventory snapshot captured at notebook start.,outputs\22_stable_diffusion_classical_metrics\...,True,1693039
1,batch0_validation,batch0,csv,True,Batch 0 validation checks.,outputs\22_stable_diffusion_classical_metrics\...,False,0
2,stage_manifest,batch0_to_final,json,True,Notebook-wide stage manifest updated during th...,outputs\22_stable_diffusion_classical_metrics\...,False,0
3,metric_input_cases,batch1,csv,True,Standardized candidate-level metric input table.,outputs\22_stable_diffusion_classical_metrics\...,False,0
4,classical_metrics,batch3,csv,True,Main candidate-level classical metrics table.,outputs\22_stable_diffusion_classical_metrics\...,False,0
5,classical_metrics_summary,batch4,csv,True,Single consolidated summary table with summary...,outputs\22_stable_diffusion_classical_metrics\...,False,0
6,selected_cases,batch4,csv,True,Representative cases for visual/diagnostic rev...,outputs\22_stable_diffusion_classical_metrics\...,False,0
7,final_validation,batch5,csv,True,End-of-notebook validation checks.,outputs\22_stable_diffusion_classical_metrics\...,False,0
8,artifact_index,batch5,csv,True,Final index of important notebook outputs.,outputs\22_stable_diffusion_classical_metrics\...,False,0
9,handoff_manifest,batch5,json,True,Final manifest for downstream notebooks.,outputs\22_stable_diffusion_classical_metrics\...,False,0


In [6]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def rel(path: Path) -> str:
    return path.resolve().relative_to(PROJECT_ROOT).as_posix()


def validation_row(check_name: str, observed, expected, passed: bool, failure_message: str = "") -> dict:
    return {
        "check_name": check_name,
        "observed": observed,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
    }


required_upstream_missing_df = upstream_artifact_df.loc[
    upstream_artifact_df["required"].astype(bool) & ~upstream_artifact_df["exists"].astype(bool)
].copy()

stage_manifest = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_title": NOTEBOOK_TITLE,
    "created_at_utc": utc_now_iso(),
    "project_root": str(PROJECT_ROOT),
    "output_root": rel(OUTPUT_ROOT),
    "helper": {
        "module": "restoration_eval.metrics_classical",
        "metric_version": metrics_classical.METRIC_VERSION,
        "base_regions": list(metrics_classical.BASE_REGIONS),
        "masked_case_regions": list(metrics_classical.MASKED_CASE_REGIONS),
        "all_regions": list(metrics_classical.ALL_EVALUATION_REGIONS),
    },
    "expected_counts_from_notebook21": {
        "candidate_rows": 945,
        "zero_control_rows": 94,
        "nonzero_rows": 851,
        "expected_metric_rows": 5294,
        "formula": "94 zero controls * 2 base regions + 851 non-zero candidates * 6 regions",
    },
    "required_upstream_artifacts": upstream_artifact_df.to_dict(orient="records"),
    "planned_output_artifacts": planned_output_artifact_df.to_dict(orient="records"),
    "batch_plan": [
        {"batch": "batch0", "purpose": "setup, paths, inventory snapshot, helper version/API check"},
        {"batch": "batch1", "purpose": "load Notebook 21 handoff and build standardized metric input cases"},
        {"batch": "batch2", "purpose": "smoke-test classical metrics on representative candidates"},
        {"batch": "batch3", "purpose": "compute full candidate-level classical metrics"},
        {"batch": "batch4", "purpose": "summaries, rankings, selected cases, diagnostic figures"},
        {"batch": "batch5", "purpose": "final validation, artifact index, handoff manifest"},
    ],
}

STAGE_MANIFEST_PATH.write_text(json.dumps(stage_manifest, indent=2), encoding="utf-8")

batch0_validation_rows = [
    validation_row(
        "batch0_project_root_detected",
        str(PROJECT_ROOT),
        "project root containing src/restoration_eval and tools/build_project_inventory.py",
        (PROJECT_ROOT / "src" / "restoration_eval").is_dir() and INVENTORY_SCRIPT_PATH.is_file(),
        "Project root detection failed.",
    ),
    validation_row(
        "batch0_output_root_is_correct_notebook_folder",
        rel(OUTPUT_ROOT),
        f"outputs/{NOTEBOOK_ID}",
        rel(OUTPUT_ROOT) == f"outputs/{NOTEBOOK_ID}",
        "Notebook outputs must stay under the Notebook 22 output folder.",
    ),
    validation_row(
        "batch0_inventory_csv_exists",
        rel(INVENTORY_CSV_PATH),
        "file exists",
        INVENTORY_CSV_PATH.is_file(),
        "Project inventory CSV is missing.",
    ),
    validation_row(
        "batch0_inventory_snapshot_written",
        rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
        "file exists",
        BATCH0_INVENTORY_SNAPSHOT_PATH.is_file(),
        "Batch 0 inventory snapshot was not written.",
    ),
    validation_row(
        "batch0_helper_version",
        metrics_classical.METRIC_VERSION,
        EXPECTED_METRIC_VERSION,
        metrics_classical.METRIC_VERSION == EXPECTED_METRIC_VERSION,
        "Wrong metrics_classical helper version.",
    ),
    validation_row(
        "batch0_helper_region_policy",
        str(tuple(metrics_classical.ALL_EVALUATION_REGIONS)),
        str(EXPECTED_ALL_REGIONS),
        tuple(metrics_classical.ALL_EVALUATION_REGIONS) == EXPECTED_ALL_REGIONS,
        "Classical metric region policy does not match the final six-region policy.",
    ),
    validation_row(
        "batch0_helper_candidate_metadata_passthrough",
        str(missing_metadata_columns),
        "[]",
        len(missing_metadata_columns) == 0,
        "Helper is missing candidate-level metadata passthrough fields.",
    ),
    validation_row(
        "batch0_required_upstream_artifacts_exist",
        required_upstream_missing_df["artifact_key"].tolist(),
        "[]",
        required_upstream_missing_df.empty,
        "At least one required Notebook 21/data input artifact is missing.",
    ),
    validation_row(
        "batch0_stage_manifest_written",
        rel(STAGE_MANIFEST_PATH),
        "file exists",
        STAGE_MANIFEST_PATH.is_file(),
        "Stage manifest was not written.",
    ),
]

batch0_validation_df = pd.DataFrame(batch0_validation_rows)
batch0_validation_df.to_csv(BATCH0_VALIDATION_PATH, index=False)

print(f"Saved: {BATCH0_VALIDATION_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved: {STAGE_MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
print(f"Batch 0 checks passed: {int(batch0_validation_df['passed'].sum())} / {len(batch0_validation_df)}")

display(batch0_validation_df)

if not required_upstream_missing_df.empty:
    print("Missing required upstream artifacts:")
    display(required_upstream_missing_df[["artifact_key", "path", "notes"]])

if not batch0_validation_df["passed"].all():
    display(batch0_validation_df.loc[~batch0_validation_df["passed"], ["check_name", "failure_message"]])
    raise RuntimeError("Batch 0 validation failed. Fix the setup/input artifacts before Batch 1.")

print("Batch 0 complete. Ready for Batch 1.")


Saved: outputs\22_stable_diffusion_classical_metrics\validation\batch0_validation.csv
Saved: outputs\22_stable_diffusion_classical_metrics\manifests\stable_diffusion_classical_metrics_stage_manifest.json
Batch 0 checks passed: 9 / 9


,check_name,observed,expected,passed,failure_message
0,batch0_project_root_detected,D:\Masters\FH\Thesis\painting-restoration-eval,project root containing src/restoration_eval a...,True,
1,batch0_output_root_is_correct_notebook_folder,outputs/22_stable_diffusion_classical_metrics,outputs/22_stable_diffusion_classical_metrics,True,
2,batch0_inventory_csv_exists,outputs/inventory/project_file_inventory.csv,file exists,True,
3,batch0_inventory_snapshot_written,outputs/22_stable_diffusion_classical_metrics/...,file exists,True,
4,batch0_helper_version,2.1.0,2.1.0,True,
5,batch0_helper_region_policy,"('full_image', 'content_region', 'masked_regio...","('full_image', 'content_region', 'masked_regio...",True,
6,batch0_helper_candidate_metadata_passthrough,[],[],True,
7,batch0_required_upstream_artifacts_exist,[],[],True,
8,batch0_stage_manifest_written,outputs/22_stable_diffusion_classical_metrics/...,file exists,True,


Batch 0 complete. Ready for Batch 1.


In [7]:
# Batch 1 / Cell 6 — Load Notebook 21 handoff outputs
BATCH1_INPUT_CASES_PATH = OUTPUT_DIRS["tables"] / "stable_diffusion_classical_metric_input_cases.csv"
BATCH1_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch1_input_validation.csv"

EXPECTED_SD_CANDIDATE_ROWS = 945
EXPECTED_SD_ZERO_CONTROL_ROWS = 94
EXPECTED_SD_NONZERO_ROWS = 851
EXPECTED_SD_METRIC_ROWS = 5294
ZERO_CONTROL_MASK_TYPE = "zero_control"

UPSTREAM_PATHS = {
    artifact["artifact_key"]: artifact["path"]
    for artifact in REQUIRED_UPSTREAM_ARTIFACTS
}

NOTEBOOK21_HANDOFF_MANIFEST_PATH = UPSTREAM_PATHS["notebook21_handoff_manifest"]
NOTEBOOK21_ARTIFACT_INDEX_PATH = UPSTREAM_PATHS["notebook21_artifact_index"]
SD_CANDIDATE_MANIFEST_PATH = UPSTREAM_PATHS["sd_candidate_manifest"]
SD_RESTORATION_AUDIT_PATH = UPSTREAM_PATHS["sd_restoration_audit"]
SD_RESTORATION_VALIDATION_PATH = UPSTREAM_PATHS["sd_restoration_validation"]
METADATA_PROCESSED_CLEAN_PATH = UPSTREAM_PATHS["metadata_processed_clean"]

with NOTEBOOK21_HANDOFF_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    notebook21_handoff_manifest = json.load(handle)

notebook21_artifact_index_df = pd.read_csv(NOTEBOOK21_ARTIFACT_INDEX_PATH)
sd_candidate_manifest_df = pd.read_csv(SD_CANDIDATE_MANIFEST_PATH)
sd_restoration_audit_df = pd.read_csv(SD_RESTORATION_AUDIT_PATH)
sd_restoration_validation_df = pd.read_csv(SD_RESTORATION_VALIDATION_PATH)
metadata_processed_clean_df = pd.read_csv(METADATA_PROCESSED_CLEAN_PATH)

print("Loaded Notebook 21 handoff artifacts:")
print(f"  Candidate manifest rows: {len(sd_candidate_manifest_df):,}")
print(f"  Restoration audit rows: {len(sd_restoration_audit_df):,}")
print(f"  Restoration validation rows: {len(sd_restoration_validation_df):,}")
print(f"  Clean metadata rows: {len(metadata_processed_clean_df):,}")

display(sd_candidate_manifest_df.head(3))
display(sd_restoration_audit_df.head(3))
display(sd_restoration_validation_df.head(3))

Loaded Notebook 21 handoff artifacts:
  Candidate manifest rows: 945
  Restoration audit rows: 945
  Restoration validation rows: 945
  Clean metadata rows: 50


,case_id,painting_id,mask_id,mask_type,generator_name,generator_version,generated_at_utc,generation_action,clean_filename,clean_path,...,guidance_scale,strength,inference_size,precision,mask_binary_threshold,preserve_unmasked_pixels,zero_control_behavior,restoration_generator_name,restoration_generator_version,candidate_created_at_utc
0,p001_loss_large,p001,p001_loss_large,loss_large,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,generated,p001_clean.png,data\processed\clean\p001_clean.png,...,7.5,1.0,512,float16,128,True,copy_without_inference,notebook_21_stable_diffusion_restoration,4.2_soft_mask_nonzero_fallback,2026-08-07T12:02:18.443013+00:00
1,p001_loss_small,p001,p001_loss_small,loss_small,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,generated,p001_clean.png,data\processed\clean\p001_clean.png,...,7.5,1.0,512,float16,128,True,copy_without_inference,notebook_21_stable_diffusion_restoration,4.2_soft_mask_nonzero_fallback,2026-08-07T12:02:18.443013+00:00
2,p001_mixed_damage,p001,p001_mixed_damage,mixed_damage,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,generated,p001_clean.png,data\processed\clean\p001_clean.png,...,7.5,1.0,512,float16,128,True,copy_without_inference,notebook_21_stable_diffusion_restoration,4.2_soft_mask_nonzero_fallback,2026-08-07T12:02:18.443013+00:00


,case_id,painting_id,mask_id,mask_type,generator_name,generator_version,generated_at_utc,generation_action,clean_filename,clean_path,...,after_cuda_max_memory_allocated_bytes,python_version,operating_system,processor,torch_version,cuda_available,cuda_device_count,cuda_device_name,diffusers_version,attempt_count
0,p001_loss_large,p001,p001_loss_large,loss_large,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,generated,p001_clean.png,data\processed\clean\p001_clean.png,...,2821253632,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1
1,p001_loss_small,p001,p001_loss_small,loss_small,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,generated,p001_clean.png,data\processed\clean\p001_clean.png,...,2821253632,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1
2,p001_mixed_damage,p001,p001_mixed_damage,mixed_damage,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,generated,p001_clean.png,data\processed\clean\p001_clean.png,...,2821253632,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1


,restoration_case_id,candidate_id,prompt_policy_id,prompt_variant_id,prompt_ablation_subset,dataset_name,source_case_key,source_case_id,case_id,painting_id,...,mean_abs_diff,max_abs_diff,inside_mask_changed,inside_mask_mean_abs_diff,inside_mask_max_abs_diff,outside_mask_exact_match,outside_mask_mean_abs_diff,outside_mask_max_abs_diff,validation_passed,validation_issue
0,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,sd_inpaint_prompt_ablation_v1,p00_generic,False,canonical,canonical__p001_loss_large,p001_loss_large,p001_loss_large,p001,...,24.995058,255.0,True,213.606378,255.0,True,0.0,0.0,True,NaN
1,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,sd_inpaint_prompt_ablation_v1,p00_generic,False,canonical,canonical__p001_loss_small,p001_loss_small,p001_loss_small,p001,...,7.115802,253.0,True,230.115174,253.0,True,0.0,0.0,True,NaN
2,sd__can__p001__mixed_damage__p00_generic__s202...,sd__can__p001__mixed_damage__p00_generic__s202...,sd_inpaint_prompt_ablation_v1,p00_generic,False,canonical,canonical__p001_mixed_damage,p001_mixed_damage,p001_mixed_damage,p001,...,16.562365,255.0,True,213.606812,255.0,True,0.0,0.0,True,NaN


In [8]:
# Batch 1 / Cell 7 — Build standardized candidate-level metric input cases
def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned.columns = [str(column).strip() for column in cleaned.columns]
    return cleaned


def is_blank_series(series: pd.Series) -> pd.Series:
    return series.isna() | series.astype(str).str.strip().isin(["", "nan", "None", "NaN"])


def resolve_project_path(path_value) -> Path:
    if pd.isna(path_value) or str(path_value).strip() == "":
        return Path("")
    raw_path = Path(str(path_value))
    if raw_path.is_absolute():
        return raw_path
    return PROJECT_ROOT / raw_path


def to_project_relative_posix(path_value) -> str:
    if pd.isna(path_value) or str(path_value).strip() == "":
        return ""
    resolved_path = resolve_project_path(path_value).resolve()
    try:
        return resolved_path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return resolved_path.as_posix()


def path_exists(path_value) -> bool:
    if pd.isna(path_value) or str(path_value).strip() == "":
        return False
    return resolve_project_path(path_value).is_file()


def require_dataframe_columns(df: pd.DataFrame, required_columns: list[str], dataframe_name: str) -> list[str]:
    return [column for column in required_columns if column not in df.columns]


sd_candidate_manifest_df = normalize_column_names(sd_candidate_manifest_df)
sd_restoration_audit_df = normalize_column_names(sd_restoration_audit_df)
sd_restoration_validation_df = normalize_column_names(sd_restoration_validation_df)
metadata_processed_clean_df = normalize_column_names(metadata_processed_clean_df)

required_candidate_columns = [
    "candidate_id",
    "restoration_case_id",
    "dataset_name",
    "source_case_key",
    "case_id",
    "painting_id",
    "mask_type",
    "model_name",
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
    "prompt_policy_id",
    "prompt_variant_id",
    "prompt_template_name",
    "candidate_seed",
]

required_audit_columns = ["candidate_id", "status", "output_written"]
required_validation_columns = ["candidate_id", "validation_passed", "restored_exists"]

missing_candidate_columns = require_dataframe_columns(
    sd_candidate_manifest_df,
    required_candidate_columns,
    "sd_candidate_manifest_df",
)
missing_audit_columns = require_dataframe_columns(
    sd_restoration_audit_df,
    required_audit_columns,
    "sd_restoration_audit_df",
)
missing_validation_columns = require_dataframe_columns(
    sd_restoration_validation_df,
    required_validation_columns,
    "sd_restoration_validation_df",
)

if missing_candidate_columns or missing_audit_columns or missing_validation_columns:
    raise RuntimeError(
        "Missing required Notebook 21 columns:\n"
        f"candidate: {missing_candidate_columns}\n"
        f"audit: {missing_audit_columns}\n"
        f"validation: {missing_validation_columns}"
    )

metric_input_df = sd_candidate_manifest_df.copy()

audit_by_candidate_df = (
    sd_restoration_audit_df
    .sort_values("candidate_id")
    .drop_duplicates("candidate_id", keep="last")
    .set_index("candidate_id")
)

validation_by_candidate_df = (
    sd_restoration_validation_df
    .sort_values("candidate_id")
    .drop_duplicates("candidate_id", keep="last")
    .set_index("candidate_id")
)

for column in metrics_classical.CANDIDATE_METADATA_COLUMNS:
    if column in ["candidate_id", "clean_path", "damaged_path", "mask_path", "restored_path"]:
        continue
    if column not in metric_input_df.columns and column in audit_by_candidate_df.columns:
        metric_input_df[column] = metric_input_df["candidate_id"].map(audit_by_candidate_df[column])
    elif column in metric_input_df.columns and column in audit_by_candidate_df.columns:
        blank_mask = is_blank_series(metric_input_df[column])
        metric_input_df.loc[blank_mask, column] = metric_input_df.loc[blank_mask, "candidate_id"].map(
            audit_by_candidate_df[column]
        )

audit_keep_columns = [
    column
    for column in [
        "candidate_id",
        "status",
        "issue",
        "output_written",
        "runtime_seconds",
        "restored_sha256",
        "effective_candidate_seed",
        "execution_device",
        "cuda_device_name",
        "restoration_generator_version",
        "mask_raw_nonzero_pixel_count",
        "mask_thresholded_pixel_count",
        "effective_mask_pixel_count",
        "effective_mask_threshold_policy",
        "mask_gray_min",
        "mask_gray_max",
    ]
    if column in sd_restoration_audit_df.columns
]

validation_keep_columns = [
    column
    for column in [
        "candidate_id",
        "validation_passed",
        "validation_issue",
        "status",
        "damaged_exists",
        "mask_exists",
        "restored_exists",
        "restored_width",
        "restored_height",
        "mask_binary_threshold",
        "mask_raw_nonzero_pixel_count",
        "mask_thresholded_pixel_count",
        "effective_mask_pixel_count",
        "effective_mask_threshold_policy",
        "mask_gray_min",
        "mask_gray_max",
        "mask_pixel_count",
        "outside_mask_pixel_count",
        "exact_match_to_damaged",
        "changed_from_damaged",
        "inside_mask_changed",
        "outside_mask_exact_match",
        "mean_abs_diff",
        "max_abs_diff",
    ]
    if column in sd_restoration_validation_df.columns
]

audit_prefixed_df = sd_restoration_audit_df[audit_keep_columns].rename(
    columns={column: f"audit_{column}" for column in audit_keep_columns if column != "candidate_id"}
)
validation_prefixed_df = sd_restoration_validation_df[validation_keep_columns].rename(
    columns={column: f"validation_{column}" for column in validation_keep_columns if column != "candidate_id"}
)

metric_input_df = metric_input_df.merge(audit_prefixed_df, on="candidate_id", how="left")
metric_input_df = metric_input_df.merge(validation_prefixed_df, on="candidate_id", how="left")

content_columns = ["content_x_min", "content_y_min", "content_x_max", "content_y_max"]
missing_or_blank_content = [
    column
    for column in content_columns
    if column not in metric_input_df.columns or is_blank_series(metric_input_df[column]).any()
]

if missing_or_blank_content:
    clean_metadata_join_columns = [
        column
        for column in ["painting_id", *content_columns, "category", "title", "artist_name"]
        if column in metadata_processed_clean_df.columns
    ]
    clean_metadata_join_df = (
        metadata_processed_clean_df[clean_metadata_join_columns]
        .drop_duplicates("painting_id", keep="first")
        if "painting_id" in clean_metadata_join_columns
        else pd.DataFrame()
    )

    if not clean_metadata_join_df.empty:
        metric_input_df = metric_input_df.merge(
            clean_metadata_join_df,
            on="painting_id",
            how="left",
            suffixes=("", "_clean_metadata"),
        )

        for column in content_columns:
            fallback_column = f"{column}_clean_metadata"
            if column not in metric_input_df.columns and fallback_column in metric_input_df.columns:
                metric_input_df[column] = metric_input_df[fallback_column]
            elif fallback_column in metric_input_df.columns:
                blank_mask = is_blank_series(metric_input_df[column])
                metric_input_df.loc[blank_mask, column] = metric_input_df.loc[blank_mask, fallback_column]

        for column in ["category", "title", "artist_name"]:
            fallback_column = f"{column}_clean_metadata"
            if fallback_column in metric_input_df.columns and column in metric_input_df.columns:
                blank_mask = is_blank_series(metric_input_df[column])
                metric_input_df.loc[blank_mask, column] = metric_input_df.loc[blank_mask, fallback_column]

for path_column in ["clean_path", "damaged_path", "mask_path", "restored_path"]:
    metric_input_df[path_column] = metric_input_df[path_column].map(to_project_relative_posix)
    metric_input_df[f"{path_column}_exists"] = metric_input_df[path_column].map(path_exists)

metric_input_df["metric_case_id"] = metric_input_df["candidate_id"]
metric_input_df["metric_input_source_notebook"] = "21_stable_diffusion_restoration"
metric_input_df["metric_input_builder_notebook"] = NOTEBOOK_ID
metric_input_df["metric_input_built_at_utc"] = utc_now_iso()
metric_input_df["metric_helper_version_expected"] = metrics_classical.METRIC_VERSION

for column in content_columns:
    metric_input_df[column] = pd.to_numeric(metric_input_df[column], errors="coerce")

preferred_front_columns = [
    "metric_case_id",
    "dataset_name",
    "case_id",
    "restoration_case_id",
    "candidate_id",
    "source_case_key",
    "source_case_id",
    "source_case_id_original",
    "painting_id",
    "category",
    "title",
    "mask_id",
    "mask_type",
    "model_name",
    "prompt_policy_id",
    "prompt_variant_id",
    "prompt_template_name",
    "prompt_ablation_subset",
    "candidate_index",
    "candidate_seed",
    "effective_candidate_seed",
    "inference_mode",
    "execution_device",
    "cuda_device_name",
    "restoration_generator_version",
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
    "clean_path_exists",
    "damaged_path_exists",
    "mask_path_exists",
    "restored_path_exists",
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
]

ordered_columns = [
    column for column in preferred_front_columns if column in metric_input_df.columns
] + [
    column for column in metric_input_df.columns if column not in preferred_front_columns
]

metric_input_df = metric_input_df[ordered_columns].sort_values(
    ["dataset_name", "source_case_key", "prompt_variant_id", "candidate_seed", "candidate_id"],
    kind="stable",
).reset_index(drop=True)

display(metric_input_df.head(10))
print(f"Metric input candidate rows: {len(metric_input_df):,}")

,metric_case_id,dataset_name,case_id,restoration_case_id,candidate_id,source_case_key,source_case_id,source_case_id_original,painting_id,category,...,content_x_min_clean_metadata,content_y_min_clean_metadata,content_x_max_clean_metadata,content_y_max_clean_metadata,category_clean_metadata,title_clean_metadata,metric_input_source_notebook,metric_input_builder_notebook,metric_input_built_at_utc,metric_helper_version_expected
0,sd__can__p001__loss_large__p00_generic__s2026_...,canonical,p001_loss_large,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,canonical__p001_loss_large,p001_loss_large,p001_loss_large,p001,portrait_figure,...,52,0,715,768,portrait_figure,Juan de Pareja,21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
1,sd__can__p001__loss_small__p00_generic__s2026_...,canonical,p001_loss_small,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,canonical__p001_loss_small,p001_loss_small,p001_loss_small,p001,portrait_figure,...,52,0,715,768,portrait_figure,Juan de Pareja,21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
2,sd__can__p001__mixed_damage__p00_generic__s202...,canonical,p001_mixed_damage,sd__can__p001__mixed_damage__p00_generic__s202...,sd__can__p001__mixed_damage__p00_generic__s202...,canonical__p001_mixed_damage,p001_mixed_damage,p001_mixed_damage,p001,portrait_figure,...,52,0,715,768,portrait_figure,Juan de Pareja,21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
3,sd__can__p001__scratch_thin__p00_generic__s202...,canonical,p001_scratch_thin,sd__can__p001__scratch_thin__p00_generic__s202...,sd__can__p001__scratch_thin__p00_generic__s202...,canonical__p001_scratch_thin,p001_scratch_thin,p001_scratch_thin,p001,portrait_figure,...,52,0,715,768,portrait_figure,Juan de Pareja,21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
4,sd__can__p001__zero__p00_generic__s2026__3be10...,canonical,p001_zero_control,sd__can__p001__zero__p00_generic__s2026__3be10...,sd__can__p001__zero__p00_generic__s2026__3be10...,canonical__p001_zero_control,p001_zero_control,p001_zero_control,p001,portrait_figure,...,52,0,715,768,portrait_figure,Juan de Pareja,21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
5,sd__can__p002__loss_large__p00_generic__s2026_...,canonical,p002_loss_large,sd__can__p002__loss_large__p00_generic__s2026_...,sd__can__p002__loss_large__p00_generic__s2026_...,canonical__p002_loss_large,p002_loss_large,p002_loss_large,p002,portrait_figure,...,159,0,608,768,portrait_figure,Madame X (Madame Pierre Gautreau),21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
6,sd__can__p002__loss_small__p00_generic__s2026_...,canonical,p002_loss_small,sd__can__p002__loss_small__p00_generic__s2026_...,sd__can__p002__loss_small__p00_generic__s2026_...,canonical__p002_loss_small,p002_loss_small,p002_loss_small,p002,portrait_figure,...,159,0,608,768,portrait_figure,Madame X (Madame Pierre Gautreau),21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
7,sd__can__p002__mixed_damage__p00_generic__s202...,canonical,p002_mixed_damage,sd__can__p002__mixed_damage__p00_generic__s202...,sd__can__p002__mixed_damage__p00_generic__s202...,canonical__p002_mixed_damage,p002_mixed_damage,p002_mixed_damage,p002,portrait_figure,...,159,0,608,768,portrait_figure,Madame X (Madame Pierre Gautreau),21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
8,sd__can__p002__scratch_thin__p00_generic__s202...,canonical,p002_scratch_thin,sd__can__p002__scratch_thin__p00_generic__s202...,sd__can__p002__scratch_thin__p0

Metric input candidate rows: 945


In [9]:
# Batch 1 / Cell 8 — Validate and write Batch 1 outputs
candidate_id_set = set(sd_candidate_manifest_df["candidate_id"].astype(str))
audit_id_set = set(sd_restoration_audit_df["candidate_id"].astype(str))
validation_id_set = set(sd_restoration_validation_df["candidate_id"].astype(str))
metric_input_id_set = set(metric_input_df["candidate_id"].astype(str))

zero_control_count = int(metric_input_df["mask_type"].astype(str).eq(ZERO_CONTROL_MASK_TYPE).sum())
nonzero_count = int(len(metric_input_df) - zero_control_count)

expected_region_counts = metrics_classical.expected_region_counts_from_metadata(metric_input_df)
expected_metric_rows = metrics_classical.expected_metric_rows_from_metadata(metric_input_df)

helper_required_metric_input_columns = [
    "case_id",
    "painting_id",
    "model_name",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
]

missing_metric_input_columns = [
    column for column in helper_required_metric_input_columns
    if column not in metric_input_df.columns
]

path_existence_columns = ["clean_path_exists", "damaged_path_exists", "mask_path_exists", "restored_path_exists"]
missing_path_counts = {
    column: int((~metric_input_df[column].astype(bool)).sum())
    for column in path_existence_columns
    if column in metric_input_df.columns
}

audit_status_counts = (
    sd_restoration_audit_df["status"].astype(str).value_counts(dropna=False).to_dict()
    if "status" in sd_restoration_audit_df.columns
    else {}
)

validation_passed_counts = (
    sd_restoration_validation_df["validation_passed"].astype(bool).value_counts(dropna=False).to_dict()
    if "validation_passed" in sd_restoration_validation_df.columns
    else {}
)

batch1_validation_rows = [
    validation_row(
        "batch1_candidate_manifest_rows",
        len(sd_candidate_manifest_df),
        EXPECTED_SD_CANDIDATE_ROWS,
        len(sd_candidate_manifest_df) == EXPECTED_SD_CANDIDATE_ROWS,
        "Notebook 21 candidate manifest row count is not 945.",
    ),
    validation_row(
        "batch1_restoration_audit_rows",
        len(sd_restoration_audit_df),
        EXPECTED_SD_CANDIDATE_ROWS,
        len(sd_restoration_audit_df) == EXPECTED_SD_CANDIDATE_ROWS,
        "Notebook 21 restoration audit row count does not match expected candidate count.",
    ),
    validation_row(
        "batch1_restoration_validation_rows",
        len(sd_restoration_validation_df),
        EXPECTED_SD_CANDIDATE_ROWS,
        len(sd_restoration_validation_df) == EXPECTED_SD_CANDIDATE_ROWS,
        "Notebook 21 restoration validation row count does not match expected candidate count.",
    ),
    validation_row(
        "batch1_candidate_ids_unique",
        int(sd_candidate_manifest_df["candidate_id"].duplicated().sum()),
        0,
        int(sd_candidate_manifest_df["candidate_id"].duplicated().sum()) == 0,
        "Candidate manifest has duplicate candidate_id values.",
    ),
    validation_row(
        "batch1_audit_candidate_ids_unique",
        int(sd_restoration_audit_df["candidate_id"].duplicated().sum()),
        0,
        int(sd_restoration_audit_df["candidate_id"].duplicated().sum()) == 0,
        "Restoration audit has duplicate candidate_id values.",
    ),
    validation_row(
        "batch1_validation_candidate_ids_unique",
        int(sd_restoration_validation_df["candidate_id"].duplicated().sum()),
        0,
        int(sd_restoration_validation_df["candidate_id"].duplicated().sum()) == 0,
        "Restoration validation has duplicate candidate_id values.",
    ),
    validation_row(
        "batch1_candidate_audit_validation_id_sets_match",
        {
            "candidate_minus_audit": len(candidate_id_set - audit_id_set),
            "audit_minus_candidate": len(audit_id_set - candidate_id_set),
            "candidate_minus_validation": len(candidate_id_set - validation_id_set),
            "validation_minus_candidate": len(validation_id_set - candidate_id_set),
        },
        "all zero",
        candidate_id_set == audit_id_set == validation_id_set,
        "Candidate, audit, and validation candidate_id sets do not match.",
    ),
    validation_row(
        "batch1_audit_status_ok",
        audit_status_counts,
        {"ok": EXPECTED_SD_CANDIDATE_ROWS},
        audit_status_counts == {"ok": EXPECTED_SD_CANDIDATE_ROWS},
        "Notebook 21 audit status is not all ok.",
    ),
    validation_row(
        "batch1_validation_passed",
        validation_passed_counts,
        {True: EXPECTED_SD_CANDIDATE_ROWS},
        validation_passed_counts == {True: EXPECTED_SD_CANDIDATE_ROWS},
        "Notebook 21 validation did not pass for every candidate.",
    ),
    validation_row(
        "batch1_zero_control_rows",
        zero_control_count,
        EXPECTED_SD_ZERO_CONTROL_ROWS,
        zero_control_count == EXPECTED_SD_ZERO_CONTROL_ROWS,
        "Zero-control candidate count changed.",
    ),
    validation_row(
        "batch1_nonzero_rows",
        nonzero_count,
        EXPECTED_SD_NONZERO_ROWS,
        nonzero_count == EXPECTED_SD_NONZERO_ROWS,
        "Non-zero candidate count changed.",
    ),
    validation_row(
        "batch1_metric_input_rows",
        len(metric_input_df),
        EXPECTED_SD_CANDIDATE_ROWS,
        len(metric_input_df) == EXPECTED_SD_CANDIDATE_ROWS,
        "Metric input table does not contain one row per SD candidate.",
    ),
    validation_row(
        "batch1_metric_input_candidate_ids_match",
        len(candidate_id_set.symmetric_difference(metric_input_id_set)),
        0,
        candidate_id_set == metric_input_id_set,
        "Metric input candidate_id set does not match Notebook 21 candidate manifest.",
    ),
    validation_row(
        "batch1_metric_input_required_columns_present",
        missing_metric_input_columns,
        [],
        len(missing_metric_input_columns) == 0,
        "Metric input table is missing columns required by metrics_classical.",
    ),
    validation_row(
        "batch1_metric_input_paths_exist",
        missing_path_counts,
        "all zero",
        all(count == 0 for count in missing_path_counts.values()),
        "At least one clean/damaged/mask/restored path is missing on disk.",
    ),
    validation_row(
        "batch1_expected_region_counts",
        expected_region_counts,
        {
            "full_image": EXPECTED_SD_CANDIDATE_ROWS,
            "content_region": EXPECTED_SD_CANDIDATE_ROWS,
            "masked_region": EXPECTED_SD_NONZERO_ROWS,
            "mask_bbox_crop": EXPECTED_SD_NONZERO_ROWS,
            "boundary_region": EXPECTED_SD_NONZERO_ROWS,
            "outside_mask_region": EXPECTED_SD_NONZERO_ROWS,
        },
        expected_region_counts == {
            "full_image": EXPECTED_SD_CANDIDATE_ROWS,
            "content_region": EXPECTED_SD_CANDIDATE_ROWS,
            "masked_region": EXPECTED_SD_NONZERO_ROWS,
            "mask_bbox_crop": EXPECTED_SD_NONZERO_ROWS,
            "boundary_region": EXPECTED_SD_NONZERO_ROWS,
            "outside_mask_region": EXPECTED_SD_NONZERO_ROWS,
        },
        "Expected region counts do not match the Notebook 22 output contract.",
    ),
    validation_row(
        "batch1_expected_metric_rows",
        expected_metric_rows,
        EXPECTED_SD_METRIC_ROWS,
        expected_metric_rows == EXPECTED_SD_METRIC_ROWS,
        "Expected classical metric row count is not 5294.",
    ),
]

batch1_validation_df = pd.DataFrame(batch1_validation_rows)

metric_input_df.to_csv(BATCH1_INPUT_CASES_PATH, index=False)
batch1_validation_df.to_csv(BATCH1_VALIDATION_PATH, index=False)

stage_manifest["batch1"] = {
    "completed_at_utc": utc_now_iso(),
    "input_rows": int(len(metric_input_df)),
    "zero_control_rows": int(zero_control_count),
    "nonzero_rows": int(nonzero_count),
    "expected_metric_rows": int(expected_metric_rows),
    "expected_region_counts": {key: int(value) for key, value in expected_region_counts.items()},
    "outputs": {
        "metric_input_cases": rel(BATCH1_INPUT_CASES_PATH),
        "batch1_validation": rel(BATCH1_VALIDATION_PATH),
    },
}

STAGE_MANIFEST_PATH.write_text(json.dumps(stage_manifest, indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH1_INPUT_CASES_PATH)}")
print(f"Saved: {rel(BATCH1_VALIDATION_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 1 checks passed: {int(batch1_validation_df['passed'].sum())} / {len(batch1_validation_df)}")

display(batch1_validation_df)

if not batch1_validation_df["passed"].all():
    display(batch1_validation_df.loc[~batch1_validation_df["passed"], ["check_name", "failure_message"]])
    raise RuntimeError("Batch 1 validation failed. Fix input standardization before Batch 2.")

Saved: outputs/22_stable_diffusion_classical_metrics/tables/stable_diffusion_classical_metric_input_cases.csv
Saved: outputs/22_stable_diffusion_classical_metrics/validation/batch1_input_validation.csv
Updated: outputs/22_stable_diffusion_classical_metrics/manifests/stable_diffusion_classical_metrics_stage_manifest.json
Batch 1 checks passed: 17 / 17


,check_name,observed,expected,passed,failure_message
0,batch1_candidate_manifest_rows,945,945,True,
1,batch1_restoration_audit_rows,945,945,True,
2,batch1_restoration_validation_rows,945,945,True,
3,batch1_candidate_ids_unique,0,0,True,
4,batch1_audit_candidate_ids_unique,0,0,True,
5,batch1_validation_candidate_ids_unique,0,0,True,
6,batch1_candidate_audit_validation_id_sets_match,"{'candidate_minus_audit': 0, 'audit_minus_cand...",all zero,True,
7,batch1_audit_status_ok,{'ok': 945},{'ok': 945},True,
8,batch1_validation_passed,{True: 945},{True: 945},True,
9,batch1_zero_control_rows,94,94,True,


In [10]:
# Batch 2 / Cell 9 — Load Batch 1 metric input and select representative smoke cases
BATCH2_SMOKE_METRICS_PATH = OUTPUT_DIRS["validation"] / "stable_diffusion_classical_metrics_smoke.csv"
BATCH2_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch2_smoke_validation.csv"

SMOKE_ZERO_CONTROL_TARGET = 2
SMOKE_GENERIC_TARGET = 2
SMOKE_CONTEXTUAL_TARGET = 2
SMOKE_TOTAL_TARGET = SMOKE_ZERO_CONTROL_TARGET + SMOKE_GENERIC_TARGET + SMOKE_CONTEXTUAL_TARGET

if not BATCH1_INPUT_CASES_PATH.is_file():
    raise FileNotFoundError(
        f"Batch 1 metric input table is missing: {BATCH1_INPUT_CASES_PATH}. "
        "Run Batch 1 before Batch 2."
    )

metric_input_cases_df = pd.read_csv(BATCH1_INPUT_CASES_PATH)

required_smoke_columns = [
    "candidate_id",
    "case_id",
    "painting_id",
    "model_name",
    "mask_type",
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
]

missing_smoke_input_columns = [
    column for column in required_smoke_columns
    if column not in metric_input_cases_df.columns
]

if missing_smoke_input_columns:
    raise RuntimeError(
        f"Batch 1 metric input table is missing required smoke-test columns: {missing_smoke_input_columns}"
    )


def lower_text_column(df: pd.DataFrame, column: str) -> pd.Series:
    if column not in df.columns:
        return pd.Series([""] * len(df), index=df.index)
    return df[column].fillna("").astype(str).str.lower()


prompt_text_for_selection = (
    lower_text_column(metric_input_cases_df, "prompt_policy_id")
    + " "
    + lower_text_column(metric_input_cases_df, "prompt_variant_id")
    + " "
    + lower_text_column(metric_input_cases_df, "prompt_template_name")
    + " "
    + lower_text_column(metric_input_cases_df, "prompt_ablation_subset")
)

zero_control_mask = metric_input_cases_df["mask_type"].fillna("").astype(str).eq(ZERO_CONTROL_MASK_TYPE)

generic_prompt_mask = (
    ~zero_control_mask
    & prompt_text_for_selection.str.contains("generic|baseline|default", regex=True, na=False)
)

contextual_prompt_mask = (
    ~zero_control_mask
    & ~generic_prompt_mask
)

sort_columns = [
    column for column in [
        "dataset_name",
        "source_case_key",
        "case_id",
        "prompt_variant_id",
        "candidate_seed",
        "candidate_id",
    ]
    if column in metric_input_cases_df.columns
]


def select_distinct_smoke_rows(candidate_df: pd.DataFrame, target_count: int) -> pd.DataFrame:
    if candidate_df.empty or target_count <= 0:
        return candidate_df.head(0).copy()

    sorted_df = candidate_df.sort_values(sort_columns, kind="stable") if sort_columns else candidate_df.copy()

    distinct_key_columns = [
        column for column in ["source_case_key", "case_id", "painting_id", "mask_type"]
        if column in sorted_df.columns
    ]

    if distinct_key_columns:
        selected_df = sorted_df.drop_duplicates(distinct_key_columns, keep="first").head(target_count)
        if len(selected_df) < target_count:
            remainder_df = sorted_df.loc[~sorted_df["candidate_id"].isin(selected_df["candidate_id"])]
            selected_df = pd.concat(
                [selected_df, remainder_df.head(target_count - len(selected_df))],
                ignore_index=True,
            )
        return selected_df.head(target_count).copy()

    return sorted_df.head(target_count).copy()


smoke_zero_df = select_distinct_smoke_rows(
    metric_input_cases_df.loc[zero_control_mask],
    SMOKE_ZERO_CONTROL_TARGET,
)

smoke_generic_df = select_distinct_smoke_rows(
    metric_input_cases_df.loc[generic_prompt_mask],
    SMOKE_GENERIC_TARGET,
)

smoke_contextual_df = select_distinct_smoke_rows(
    metric_input_cases_df.loc[contextual_prompt_mask],
    SMOKE_CONTEXTUAL_TARGET,
)

smoke_input_df = (
    pd.concat([smoke_zero_df, smoke_generic_df, smoke_contextual_df], ignore_index=True)
    .drop_duplicates("candidate_id", keep="first")
    .reset_index(drop=True)
)

smoke_input_df["smoke_selection_group"] = [
    *["zero_control"] * len(smoke_zero_df.drop_duplicates("candidate_id")),
    *["generic_prompt"] * len(smoke_generic_df.drop_duplicates("candidate_id")),
    *["contextual_prompt"] * len(smoke_contextual_df.drop_duplicates("candidate_id")),
][: len(smoke_input_df)]

print(f"Batch 1 metric input rows: {len(metric_input_cases_df):,}")
print(f"Smoke input rows selected: {len(smoke_input_df):,}")
print(smoke_input_df["smoke_selection_group"].value_counts(dropna=False).to_string())

display_columns = [
    column for column in [
        "smoke_selection_group",
        "dataset_name",
        "case_id",
        "restoration_case_id",
        "candidate_id",
        "mask_type",
        "prompt_policy_id",
        "prompt_variant_id",
        "prompt_template_name",
        "candidate_seed",
    ]
    if column in smoke_input_df.columns
]

display(smoke_input_df[display_columns])

Batch 1 metric input rows: 945
Smoke input rows selected: 6
smoke_selection_group
zero_control         2
generic_prompt       2
contextual_prompt    2


,smoke_selection_group,dataset_name,case_id,restoration_case_id,candidate_id,mask_type,prompt_policy_id,prompt_variant_id,prompt_template_name,candidate_seed
0,zero_control,canonical,p001_zero_control,sd__can__p001__zero__p00_generic__s2026__3be10...,sd__can__p001__zero__p00_generic__s2026__3be10...,zero_control,sd_inpaint_prompt_ablation_v1,p00_generic,generic_restoration,2026
1,zero_control,canonical,p002_zero_control,sd__can__p002__zero__p00_generic__s2026__606dc...,sd__can__p002__zero__p00_generic__s2026__606dc...,zero_control,sd_inpaint_prompt_ablation_v1,p00_generic,generic_restoration,2026
2,generic_prompt,canonical,p001_loss_large,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,loss_large,sd_inpaint_prompt_ablation_v1,p00_generic,generic_restoration,2026
3,generic_prompt,canonical,p001_loss_small,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,generic_restoration,2026
4,contextual_prompt,canonical,p004_loss_large,sd__can__p004__loss_large__p01_style_period__s...,sd__can__p004__loss_large__p01_style_period__s...,loss_large,sd_inpaint_prompt_ablation_v1,p01_style_period,style_period_context,2026
5,contextual_prompt,canonical,p004_loss_small,sd__can__p004__loss_small__p01_style_period__s...,sd__can__p004__loss_small__p01_style_period__s...,loss_small,sd_inpaint_prompt_ablation_v1,p01_style_period,style_period_context,2026


In [11]:
# Batch 2 / Cell 10 — Compute smoke classical metrics with the real helper
if smoke_input_df.empty:
    raise RuntimeError("Smoke input table is empty. Batch 2 cannot continue.")

smoke_expected_region_counts = metrics_classical.expected_region_counts_from_metadata(smoke_input_df)
smoke_expected_metric_rows = metrics_classical.expected_metric_rows_from_metadata(smoke_input_df)

print("Smoke expected region counts:")
print(json.dumps({key: int(value) for key, value in smoke_expected_region_counts.items()}, indent=2))
print(f"Smoke expected metric rows: {smoke_expected_metric_rows}")

smoke_metrics_df = metrics_classical.compute_classical_metrics_for_restorations(
    smoke_input_df,
    target_size=768,
    mask_bbox_margin=8,
    boundary_width=3,
    progress_every=1,
)

print(f"Smoke metric rows produced: {len(smoke_metrics_df):,}")
display(smoke_metrics_df.head(12))

Smoke expected region counts:
{
  "full_image": 6,
  "content_region": 6,
  "masked_region": 4,
  "mask_bbox_crop": 4,
  "boundary_region": 4,
  "outside_mask_region": 4
}
Smoke expected metric rows: 28
Starting classical metric computation
  Cases: 6
  Target shape: (768, 768)
  Mask bbox margin: 8
  Boundary width: 3
Computing case 1/6 (p001_loss_large) | elapsed 0.00s
Computing case 2/6 (p001_loss_small) | elapsed 3.77s
Computing case 3/6 (p001_zero_control) | elapsed 6.05s
Computing case 4/6 (p002_zero_control) | elapsed 7.64s
Computing case 5/6 (p004_loss_large) | elapsed 9.00s
Computing case 6/6 (p004_loss_small) | elapsed 10.91s
Classical metric computation complete
  Runtime: 13.16 seconds
  Output rows: 28
  Region counts:
evaluation_region
full_image             6
content_region         6
masked_region          4
boundary_region        4
outside_mask_region    4
mask_bbox_crop         4
  Status counts:
status
ok    28
Smoke metric rows produced: 28


,dataset_name,case_id,painting_id,category,title,mask_id,mask_type,model_name,restoration_case_id,candidate_id,...,ssim_improvement,metric_module,metric_version,metric_timestamp_utc,python_version,numpy_version,pandas_version,skimage_version,status,issue
0,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,0.068244,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:00:11.376084+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
1,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,0.079151,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:00:11.376084+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
2,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:00:11.376084+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
3,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:00:11.376084+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
4,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:00:11.376084+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
5,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,0.329429,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:00:11.376084+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
6,canonical,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,stable_diffusion_inpainting,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,...,0.031794,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:00:11.376084+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
7,canonical,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,stable_diffusion_inpainting,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,...,0.036875,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:00:11.376084+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
8,canonical,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,stable_diffusion_inpainting,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:00:11.376084+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
9,canonical,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,stable_diffusion_inpainting,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:00:11.376084+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,


In [12]:
# Batch 2 / Cell 11 — Validate and write Batch 2 outputs
helper_smoke_validation_df = metrics_classical.validate_classical_metrics(
    smoke_metrics_df,
    expected_rows=smoke_expected_metric_rows,
    expected_region_counts=smoke_expected_region_counts,
    key_columns=("candidate_id", "evaluation_region"),
)

helper_smoke_validation_df = helper_smoke_validation_df.rename(
    columns={"check": "helper_check_name", "detail": "helper_detail"}
)

smoke_region_counts = smoke_metrics_df["evaluation_region"].value_counts().to_dict()
smoke_status_counts = smoke_metrics_df["status"].value_counts(dropna=False).to_dict()

required_smoke_output_columns = [
    "candidate_id",
    "restoration_case_id",
    "prompt_variant_id",
    "evaluation_region",
    "damaged_mse",
    "restored_mse",
    "mse_improvement",
    "damaged_mae",
    "restored_mae",
    "mae_improvement",
    "metric_version",
    "status",
]

missing_smoke_output_columns = [
    column for column in required_smoke_output_columns
    if column not in smoke_metrics_df.columns
]

candidate_metadata_missing_count = (
    int(smoke_metrics_df["candidate_id"].isna().sum())
    if "candidate_id" in smoke_metrics_df.columns
    else len(smoke_metrics_df)
)

prompt_metadata_missing_count = (
    int(smoke_metrics_df["prompt_variant_id"].isna().sum())
    if "prompt_variant_id" in smoke_metrics_df.columns
    else len(smoke_metrics_df)
)

duplicate_candidate_region_rows = (
    int(smoke_metrics_df.duplicated(["candidate_id", "evaluation_region"], keep=False).sum())
    if {"candidate_id", "evaluation_region"}.issubset(smoke_metrics_df.columns)
    else len(smoke_metrics_df)
)

smoke_selection_group_counts = smoke_input_df["smoke_selection_group"].value_counts(dropna=False).to_dict()

batch2_validation_rows = [
    validation_row(
        "batch2_smoke_input_not_empty",
        len(smoke_input_df),
        f">= {SMOKE_TOTAL_TARGET}",
        len(smoke_input_df) >= SMOKE_TOTAL_TARGET,
        "Smoke input selection produced too few rows.",
    ),
    validation_row(
        "batch2_smoke_zero_control_cases_present",
        int((smoke_input_df["smoke_selection_group"] == "zero_control").sum()),
        f">= {SMOKE_ZERO_CONTROL_TARGET}",
        int((smoke_input_df["smoke_selection_group"] == "zero_control").sum()) >= SMOKE_ZERO_CONTROL_TARGET,
        "Smoke selection does not include enough zero-control cases.",
    ),
    validation_row(
        "batch2_smoke_generic_cases_present",
        int((smoke_input_df["smoke_selection_group"] == "generic_prompt").sum()),
        f">= {SMOKE_GENERIC_TARGET}",
        int((smoke_input_df["smoke_selection_group"] == "generic_prompt").sum()) >= SMOKE_GENERIC_TARGET,
        "Smoke selection does not include enough generic prompt cases.",
    ),
    validation_row(
        "batch2_smoke_contextual_cases_present",
        int((smoke_input_df["smoke_selection_group"] == "contextual_prompt").sum()),
        f">= {SMOKE_CONTEXTUAL_TARGET}",
        int((smoke_input_df["smoke_selection_group"] == "contextual_prompt").sum()) >= SMOKE_CONTEXTUAL_TARGET,
        "Smoke selection does not include enough contextual/non-generic prompt cases.",
    ),
    validation_row(
        "batch2_smoke_expected_metric_rows_positive",
        int(smoke_expected_metric_rows),
        "> 0",
        int(smoke_expected_metric_rows) > 0,
        "Smoke expected metric row count is not positive.",
    ),
    validation_row(
        "batch2_smoke_metric_rows_match_expected",
        len(smoke_metrics_df),
        int(smoke_expected_metric_rows),
        len(smoke_metrics_df) == int(smoke_expected_metric_rows),
        "Smoke metric output row count does not match helper expectation.",
    ),
    validation_row(
        "batch2_smoke_metric_status_ok",
        smoke_status_counts,
        {"ok": int(smoke_expected_metric_rows)},
        smoke_status_counts == {"ok": int(smoke_expected_metric_rows)},
        "Smoke metric computation produced warning/error rows.",
    ),
    validation_row(
        "batch2_smoke_required_output_columns_present",
        missing_smoke_output_columns,
        [],
        len(missing_smoke_output_columns) == 0,
        "Smoke metric output is missing required columns.",
    ),
    validation_row(
        "batch2_smoke_candidate_metadata_passthrough",
        candidate_metadata_missing_count,
        0,
        candidate_metadata_missing_count == 0,
        "candidate_id was not preserved in smoke metric output.",
    ),
    validation_row(
        "batch2_smoke_prompt_metadata_passthrough",
        prompt_metadata_missing_count,
        0,
        prompt_metadata_missing_count == 0,
        "prompt_variant_id was not preserved in smoke metric output.",
    ),
    validation_row(
        "batch2_smoke_candidate_region_keys_unique",
        duplicate_candidate_region_rows,
        0,
        duplicate_candidate_region_rows == 0,
        "candidate_id + evaluation_region is not unique in smoke metric output.",
    ),
    validation_row(
        "batch2_smoke_helper_validation_passed",
        helper_smoke_validation_df["passed"].astype(bool).to_dict(),
        "all True",
        bool(helper_smoke_validation_df["passed"].astype(bool).all()),
        "metrics_classical.validate_classical_metrics failed for smoke output.",
    ),
]

batch2_validation_df = pd.DataFrame(batch2_validation_rows)

smoke_metrics_df.to_csv(BATCH2_SMOKE_METRICS_PATH, index=False)
batch2_validation_df.to_csv(BATCH2_VALIDATION_PATH, index=False)

stage_manifest["batch2"] = {
    "completed_at_utc": utc_now_iso(),
    "smoke_input_rows": int(len(smoke_input_df)),
    "smoke_selection_group_counts": {
        str(key): int(value) for key, value in smoke_selection_group_counts.items()
    },
    "expected_metric_rows": int(smoke_expected_metric_rows),
    "expected_region_counts": {
        str(key): int(value) for key, value in smoke_expected_region_counts.items()
    },
    "actual_metric_rows": int(len(smoke_metrics_df)),
    "actual_region_counts": {
        str(key): int(value) for key, value in smoke_region_counts.items()
    },
    "status_counts": {
        str(key): int(value) for key, value in smoke_status_counts.items()
    },
    "outputs": {
        "smoke_metrics": rel(BATCH2_SMOKE_METRICS_PATH),
        "batch2_validation": rel(BATCH2_VALIDATION_PATH),
    },
}

STAGE_MANIFEST_PATH.write_text(json.dumps(stage_manifest, indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH2_SMOKE_METRICS_PATH)}")
print(f"Saved: {rel(BATCH2_VALIDATION_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 2 checks passed: {int(batch2_validation_df['passed'].sum())} / {len(batch2_validation_df)}")

display(helper_smoke_validation_df)
display(batch2_validation_df)

if not batch2_validation_df["passed"].all():
    display(batch2_validation_df.loc[~batch2_validation_df["passed"], ["check_name", "failure_message"]])
    raise RuntimeError("Batch 2 validation failed. Fix smoke computation before Batch 3.")

Saved: outputs/22_stable_diffusion_classical_metrics/validation/stable_diffusion_classical_metrics_smoke.csv
Saved: outputs/22_stable_diffusion_classical_metrics/validation/batch2_smoke_validation.csv
Updated: outputs/22_stable_diffusion_classical_metrics/manifests/stable_diffusion_classical_metrics_stage_manifest.json
Batch 2 checks passed: 12 / 12


,helper_check_name,passed,helper_detail
0,required_columns,True,All required columns present.
1,row_count,True,"Expected 28, found 28."
2,no_error_rows,True,Error rows: 0; warning rows: 0.
3,unique_metric_keys,True,Rows participating in duplicate metric keys: 0.
4,region_counts,True,All evaluation-region counts match expectations.
5,positive_region_pixel_counts,True,Successful rows with non-positive region size: 0.


,check_name,observed,expected,passed,failure_message
0,batch2_smoke_input_not_empty,6,>= 6,True,
1,batch2_smoke_zero_control_cases_present,2,>= 2,True,
2,batch2_smoke_generic_cases_present,2,>= 2,True,
3,batch2_smoke_contextual_cases_present,2,>= 2,True,
4,batch2_smoke_expected_metric_rows_positive,28,> 0,True,
5,batch2_smoke_metric_rows_match_expected,28,28,True,
6,batch2_smoke_metric_status_ok,{'ok': 28},{'ok': 28},True,
7,batch2_smoke_required_output_columns_present,[],[],True,
8,batch2_smoke_candidate_metadata_passthrough,0,0,True,
9,batch2_smoke_prompt_metadata_passthrough,0,0,True,


In [13]:
# Batch 3 / Cell 12 — Load full metric input and preflight
import os
from contextlib import contextmanager

BATCH3_METRICS_PATH = OUTPUT_DIRS["metrics"] / "stable_diffusion_classical_metrics.csv"
BATCH3_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "stable_diffusion_classical_metrics_validation.csv"

BATCH3_TARGET_SIZE = 768
BATCH3_MASK_BBOX_MARGIN = 8
BATCH3_BOUNDARY_WIDTH = 3
BATCH3_PROGRESS_EVERY = 25

EXPECTED_BATCH3_OUTPUT_FILES = [
    BATCH3_METRICS_PATH,
    BATCH3_VALIDATION_PATH,
]

@contextmanager
def working_directory(path: Path):
    previous_cwd = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous_cwd)


def project_relative_path_exists(path_value) -> bool:
    if pd.isna(path_value) or str(path_value).strip() == "":
        return False

    candidate_path = Path(str(path_value))
    if candidate_path.is_absolute():
        return candidate_path.is_file()

    return (PROJECT_ROOT / candidate_path).is_file()


if not BATCH1_INPUT_CASES_PATH.is_file():
    raise FileNotFoundError(
        f"Batch 1 metric input table is missing: {BATCH1_INPUT_CASES_PATH}. "
        "Run Batch 1 before Batch 3."
    )

if not BATCH2_VALIDATION_PATH.is_file():
    raise FileNotFoundError(
        f"Batch 2 validation output is missing: {BATCH2_VALIDATION_PATH}. "
        "Run Batch 2 before Batch 3."
    )

metric_input_cases_df = pd.read_csv(BATCH1_INPUT_CASES_PATH)
batch2_validation_df = pd.read_csv(BATCH2_VALIDATION_PATH)

batch2_passed = bool(batch2_validation_df["passed"].astype(bool).all())

path_columns = ["clean_path", "damaged_path", "mask_path", "restored_path"]
missing_path_counts = {
    column: int((~metric_input_cases_df[column].map(project_relative_path_exists)).sum())
    for column in path_columns
    if column in metric_input_cases_df.columns
}

zero_control_count = int(
    metric_input_cases_df["mask_type"].astype(str).eq(ZERO_CONTROL_MASK_TYPE).sum()
)
nonzero_count = int(len(metric_input_cases_df) - zero_control_count)

print(f"Batch 2 validation passed: {batch2_passed}")
print(f"Batch 3 input rows: {len(metric_input_cases_df):,}")
print(f"Zero-control rows: {zero_control_count:,}")
print(f"Non-zero rows: {nonzero_count:,}")
print("Missing path counts:")
print(json.dumps(missing_path_counts, indent=2))

display(metric_input_cases_df.head(5))

Batch 2 validation passed: True
Batch 3 input rows: 945
Zero-control rows: 94
Non-zero rows: 851
Missing path counts:
{
  "clean_path": 0,
  "damaged_path": 0,
  "mask_path": 0,
  "restored_path": 0
}


,metric_case_id,dataset_name,case_id,restoration_case_id,candidate_id,source_case_key,source_case_id,source_case_id_original,painting_id,category,...,content_x_min_clean_metadata,content_y_min_clean_metadata,content_x_max_clean_metadata,content_y_max_clean_metadata,category_clean_metadata,title_clean_metadata,metric_input_source_notebook,metric_input_builder_notebook,metric_input_built_at_utc,metric_helper_version_expected
0,sd__can__p001__loss_large__p00_generic__s2026_...,canonical,p001_loss_large,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,canonical__p001_loss_large,p001_loss_large,p001_loss_large,p001,portrait_figure,...,52,0,715,768,portrait_figure,Juan de Pareja,21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
1,sd__can__p001__loss_small__p00_generic__s2026_...,canonical,p001_loss_small,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,canonical__p001_loss_small,p001_loss_small,p001_loss_small,p001,portrait_figure,...,52,0,715,768,portrait_figure,Juan de Pareja,21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
2,sd__can__p001__mixed_damage__p00_generic__s202...,canonical,p001_mixed_damage,sd__can__p001__mixed_damage__p00_generic__s202...,sd__can__p001__mixed_damage__p00_generic__s202...,canonical__p001_mixed_damage,p001_mixed_damage,p001_mixed_damage,p001,portrait_figure,...,52,0,715,768,portrait_figure,Juan de Pareja,21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
3,sd__can__p001__scratch_thin__p00_generic__s202...,canonical,p001_scratch_thin,sd__can__p001__scratch_thin__p00_generic__s202...,sd__can__p001__scratch_thin__p00_generic__s202...,canonical__p001_scratch_thin,p001_scratch_thin,p001_scratch_thin,p001,portrait_figure,...,52,0,715,768,portrait_figure,Juan de Pareja,21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0
4,sd__can__p001__zero__p00_generic__s2026__3be10...,canonical,p001_zero_control,sd__can__p001__zero__p00_generic__s2026__3be10...,sd__can__p001__zero__p00_generic__s2026__3be10...,canonical__p001_zero_control,p001_zero_control,p001_zero_control,p001,portrait_figure,...,52,0,715,768,portrait_figure,Juan de Pareja,21_stable_diffusion_restoration,22_stable_diffusion_classical_metrics,2026-08-07T17:50:02.616958+00:00,2.1.0


In [14]:
# Batch 3 / Cell 13 — Compute full candidate-level classical metrics
if not batch2_passed:
    raise RuntimeError("Batch 2 validation did not pass. Fix Batch 2 before running full Batch 3 metrics.")

if len(metric_input_cases_df) != EXPECTED_SD_CANDIDATE_ROWS:
    raise RuntimeError(
        f"Batch 3 expected {EXPECTED_SD_CANDIDATE_ROWS} input rows, "
        f"found {len(metric_input_cases_df)}."
    )

if any(count != 0 for count in missing_path_counts.values()):
    raise RuntimeError(
        f"Batch 3 input contains missing image paths: {missing_path_counts}"
    )

with working_directory(PROJECT_ROOT):
    batch3_expected_region_counts = metrics_classical.expected_region_counts_from_metadata(
        metric_input_cases_df
    )
    batch3_expected_metric_rows = metrics_classical.expected_metric_rows_from_metadata(
        metric_input_cases_df
    )

print("Batch 3 expected region counts:")
print(json.dumps({key: int(value) for key, value in batch3_expected_region_counts.items()}, indent=2))
print(f"Batch 3 expected metric rows: {batch3_expected_metric_rows:,}")

with working_directory(PROJECT_ROOT):
    stable_diffusion_classical_metrics_df = metrics_classical.compute_classical_metrics_for_restorations(
        metric_input_cases_df,
        target_size=BATCH3_TARGET_SIZE,
        mask_bbox_margin=BATCH3_MASK_BBOX_MARGIN,
        boundary_width=BATCH3_BOUNDARY_WIDTH,
        progress_every=BATCH3_PROGRESS_EVERY,
    )

print(f"Batch 3 metric rows produced: {len(stable_diffusion_classical_metrics_df):,}")
display(stable_diffusion_classical_metrics_df.head(10))

Batch 3 expected region counts:
{
  "full_image": 945,
  "content_region": 945,
  "masked_region": 851,
  "mask_bbox_crop": 851,
  "boundary_region": 851,
  "outside_mask_region": 851
}
Batch 3 expected metric rows: 5,294
Starting classical metric computation
  Cases: 945
  Target shape: (768, 768)
  Mask bbox margin: 8
  Boundary width: 3
Computing case 1/945 (p001_loss_large) | elapsed 0.01s
Computing case 25/945 (p004_loss_small) | elapsed 48.08s
Computing case 50/945 (p006_zero_control) | elapsed 100.66s
Computing case 75/945 (p008_scratch_thin) | elapsed 149.42s
Computing case 100/945 (p012_zero_control) | elapsed 199.70s
Computing case 125/945 (p014_scratch_thin) | elapsed 246.74s
Computing case 150/945 (p018_loss_large) | elapsed 291.58s
Computing case 175/945 (p019_zero_control) | elapsed 341.28s
Computing case 200/945 (p024_loss_large) | elapsed 390.25s
Computing case 225/945 (p025_zero_control) | elapsed 442.39s
Computing case 250/945 (p028_mixed_damage) | elapsed 492.33s
Com

,dataset_name,case_id,painting_id,category,title,mask_id,mask_type,model_name,restoration_case_id,candidate_id,...,ssim_improvement,metric_module,metric_version,metric_timestamp_utc,python_version,numpy_version,pandas_version,skimage_version,status,issue
0,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,0.068244,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
1,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,0.079151,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
2,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
3,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
4,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
5,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,0.329429,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
6,canonical,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,stable_diffusion_inpainting,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,...,0.031794,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
7,canonical,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,stable_diffusion_inpainting,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,...,0.036875,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
8,canonical,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,stable_diffusion_inpainting,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,
9,canonical,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,stable_diffusion_inpainting,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,


In [15]:
# Batch 3 / Cell 14 — Validate and write Batch 3 outputs
batch3_helper_validation_df = metrics_classical.validate_classical_metrics(
    stable_diffusion_classical_metrics_df,
    expected_rows=batch3_expected_metric_rows,
    expected_region_counts=batch3_expected_region_counts,
    key_columns=("candidate_id", "evaluation_region"),
)

batch3_helper_validation_df = batch3_helper_validation_df.rename(
    columns={"check": "helper_check_name", "detail": "helper_detail"}
)

batch3_region_counts = (
    stable_diffusion_classical_metrics_df["evaluation_region"]
    .value_counts()
    .to_dict()
)

batch3_status_counts = (
    stable_diffusion_classical_metrics_df["status"]
    .value_counts(dropna=False)
    .to_dict()
)

required_batch3_output_columns = [
    "candidate_id",
    "restoration_case_id",
    "prompt_variant_id",
    "evaluation_region",
    "region_pixel_count",
    "damaged_mse",
    "restored_mse",
    "mse_improvement",
    "damaged_mae",
    "restored_mae",
    "mae_improvement",
    "damaged_psnr",
    "restored_psnr",
    "ssim_improvement",
    "metric_module",
    "metric_version",
    "status",
    "issue",
]

missing_batch3_output_columns = [
    column for column in required_batch3_output_columns
    if column not in stable_diffusion_classical_metrics_df.columns
]

duplicate_candidate_region_rows = (
    int(
        stable_diffusion_classical_metrics_df
        .duplicated(["candidate_id", "evaluation_region"], keep=False)
        .sum()
    )
    if {"candidate_id", "evaluation_region"}.issubset(stable_diffusion_classical_metrics_df.columns)
    else len(stable_diffusion_classical_metrics_df)
)

metric_version_counts = (
    stable_diffusion_classical_metrics_df["metric_version"]
    .value_counts(dropna=False)
    .to_dict()
    if "metric_version" in stable_diffusion_classical_metrics_df.columns
    else {}
)

candidate_metadata_missing_count = (
    int(stable_diffusion_classical_metrics_df["candidate_id"].isna().sum())
    if "candidate_id" in stable_diffusion_classical_metrics_df.columns
    else len(stable_diffusion_classical_metrics_df)
)

prompt_metadata_missing_count = (
    int(stable_diffusion_classical_metrics_df["prompt_variant_id"].isna().sum())
    if "prompt_variant_id" in stable_diffusion_classical_metrics_df.columns
    else len(stable_diffusion_classical_metrics_df)
)

restoration_case_metadata_missing_count = (
    int(stable_diffusion_classical_metrics_df["restoration_case_id"].isna().sum())
    if "restoration_case_id" in stable_diffusion_classical_metrics_df.columns
    else len(stable_diffusion_classical_metrics_df)
)

fixed_output_filename_lengths = {
    path.name: len(path.name)
    for path in EXPECTED_BATCH3_OUTPUT_FILES
}

fixed_output_relative_path_lengths = {
    rel(path): len(rel(path))
    for path in EXPECTED_BATCH3_OUTPUT_FILES
}

expected_batch3_region_counts_contract = {
    "full_image": EXPECTED_SD_CANDIDATE_ROWS,
    "content_region": EXPECTED_SD_CANDIDATE_ROWS,
    "masked_region": EXPECTED_SD_NONZERO_ROWS,
    "mask_bbox_crop": EXPECTED_SD_NONZERO_ROWS,
    "boundary_region": EXPECTED_SD_NONZERO_ROWS,
    "outside_mask_region": EXPECTED_SD_NONZERO_ROWS,
}

batch3_validation_rows = [
    validation_row(
        "batch3_batch2_validation_passed",
        batch2_passed,
        True,
        batch2_passed,
        "Batch 2 validation did not pass before full metric computation.",
    ),
    validation_row(
        "batch3_input_rows",
        len(metric_input_cases_df),
        EXPECTED_SD_CANDIDATE_ROWS,
        len(metric_input_cases_df) == EXPECTED_SD_CANDIDATE_ROWS,
        "Batch 3 input does not contain one row per SD candidate.",
    ),
    validation_row(
        "batch3_zero_control_rows",
        zero_control_count,
        EXPECTED_SD_ZERO_CONTROL_ROWS,
        zero_control_count == EXPECTED_SD_ZERO_CONTROL_ROWS,
        "Zero-control candidate count changed before full metric computation.",
    ),
    validation_row(
        "batch3_nonzero_rows",
        nonzero_count,
        EXPECTED_SD_NONZERO_ROWS,
        nonzero_count == EXPECTED_SD_NONZERO_ROWS,
        "Non-zero candidate count changed before full metric computation.",
    ),
    validation_row(
        "batch3_input_paths_exist",
        missing_path_counts,
        "all zero",
        all(count == 0 for count in missing_path_counts.values()),
        "At least one clean/damaged/mask/restored path is missing on disk.",
    ),
    validation_row(
        "batch3_expected_region_counts",
        batch3_expected_region_counts,
        expected_batch3_region_counts_contract,
        batch3_expected_region_counts == expected_batch3_region_counts_contract,
        "Expected region counts do not match the Notebook 22 output contract.",
    ),
    validation_row(
        "batch3_expected_metric_rows",
        int(batch3_expected_metric_rows),
        EXPECTED_SD_METRIC_ROWS,
        int(batch3_expected_metric_rows) == EXPECTED_SD_METRIC_ROWS,
        "Expected classical metric row count is not 5294.",
    ),
    validation_row(
        "batch3_metric_rows_match_expected",
        len(stable_diffusion_classical_metrics_df),
        int(batch3_expected_metric_rows),
        len(stable_diffusion_classical_metrics_df) == int(batch3_expected_metric_rows),
        "Full metric output row count does not match helper expectation.",
    ),
    validation_row(
        "batch3_metric_status_ok",
        batch3_status_counts,
        {"ok": int(batch3_expected_metric_rows)},
        batch3_status_counts == {"ok": int(batch3_expected_metric_rows)},
        "Full metric computation produced warning/error rows.",
    ),
    validation_row(
        "batch3_required_output_columns_present",
        missing_batch3_output_columns,
        [],
        len(missing_batch3_output_columns) == 0,
        "Full metric output is missing required columns.",
    ),
    validation_row(
        "batch3_candidate_metadata_passthrough",
        candidate_metadata_missing_count,
        0,
        candidate_metadata_missing_count == 0,
        "candidate_id was not preserved in full metric output.",
    ),
    validation_row(
        "batch3_prompt_metadata_passthrough",
        prompt_metadata_missing_count,
        0,
        prompt_metadata_missing_count == 0,
        "prompt_variant_id was not preserved in full metric output.",
    ),
    validation_row(
        "batch3_restoration_case_metadata_passthrough",
        restoration_case_metadata_missing_count,
        0,
        restoration_case_metadata_missing_count == 0,
        "restoration_case_id was not preserved in full metric output.",
    ),
    validation_row(
        "batch3_candidate_region_keys_unique",
        duplicate_candidate_region_rows,
        0,
        duplicate_candidate_region_rows == 0,
        "candidate_id + evaluation_region is not unique in full metric output.",
    ),
    validation_row(
        "batch3_metric_version",
        metric_version_counts,
        {metrics_classical.METRIC_VERSION: int(batch3_expected_metric_rows)},
        metric_version_counts == {metrics_classical.METRIC_VERSION: int(batch3_expected_metric_rows)},
        "Full metric output does not consistently use the expected helper metric version.",
    ),
    validation_row(
        "batch3_helper_validation_passed",
        batch3_helper_validation_df["passed"].astype(bool).to_dict(),
        "all True",
        bool(batch3_helper_validation_df["passed"].astype(bool).all()),
        "metrics_classical.validate_classical_metrics failed for full metric output.",
    ),
    validation_row(
        "batch3_fixed_short_output_filenames",
        fixed_output_filename_lengths,
        "all <= 80 characters",
        all(length <= 80 for length in fixed_output_filename_lengths.values()),
        "Batch 3 output filenames are too long. Keep only fixed consolidated filenames.",
    ),
    validation_row(
        "batch3_fixed_short_relative_output_paths",
        fixed_output_relative_path_lengths,
        "all <= 180 characters",
        all(length <= 180 for length in fixed_output_relative_path_lengths.values()),
        "Batch 3 relative output paths are too long for cross-platform safety.",
    ),
]

batch3_validation_df = pd.DataFrame(batch3_validation_rows)

stable_diffusion_classical_metrics_df.to_csv(BATCH3_METRICS_PATH, index=False)
batch3_validation_df.to_csv(BATCH3_VALIDATION_PATH, index=False)

stage_manifest["batch3"] = {
    "completed_at_utc": utc_now_iso(),
    "input_rows": int(len(metric_input_cases_df)),
    "zero_control_rows": int(zero_control_count),
    "nonzero_rows": int(nonzero_count),
    "expected_metric_rows": int(batch3_expected_metric_rows),
    "actual_metric_rows": int(len(stable_diffusion_classical_metrics_df)),
    "expected_region_counts": {
        str(key): int(value) for key, value in batch3_expected_region_counts.items()
    },
    "actual_region_counts": {
        str(key): int(value) for key, value in batch3_region_counts.items()
    },
    "status_counts": {
        str(key): int(value) for key, value in batch3_status_counts.items()
    },
    "filename_length_policy": {
        "writes_only_consolidated_fixed_name_csvs": True,
        "no_per_candidate_or_prompt_derived_filenames": True,
        "fixed_output_filename_lengths": fixed_output_filename_lengths,
        "fixed_output_relative_path_lengths": fixed_output_relative_path_lengths,
    },
    "parameters": {
        "target_size": BATCH3_TARGET_SIZE,
        "mask_bbox_margin": BATCH3_MASK_BBOX_MARGIN,
        "boundary_width": BATCH3_BOUNDARY_WIDTH,
        "progress_every": BATCH3_PROGRESS_EVERY,
    },
    "outputs": {
        "classical_metrics": rel(BATCH3_METRICS_PATH),
        "classical_metrics_validation": rel(BATCH3_VALIDATION_PATH),
    },
}

STAGE_MANIFEST_PATH.write_text(json.dumps(stage_manifest, indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH3_METRICS_PATH)}")
print(f"Saved: {rel(BATCH3_VALIDATION_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 3 checks passed: {int(batch3_validation_df['passed'].sum())} / {len(batch3_validation_df)}")

display(batch3_helper_validation_df)
display(batch3_validation_df)

if not batch3_validation_df["passed"].all():
    display(batch3_validation_df.loc[~batch3_validation_df["passed"], ["check_name", "failure_message"]])
    raise RuntimeError("Batch 3 validation failed. Fix full metric output before Batch 4.")

Saved: outputs/22_stable_diffusion_classical_metrics/metrics/stable_diffusion_classical_metrics.csv
Saved: outputs/22_stable_diffusion_classical_metrics/validation/stable_diffusion_classical_metrics_validation.csv
Updated: outputs/22_stable_diffusion_classical_metrics/manifests/stable_diffusion_classical_metrics_stage_manifest.json
Batch 3 checks passed: 18 / 18


,helper_check_name,passed,helper_detail
0,required_columns,True,All required columns present.
1,row_count,True,"Expected 5294, found 5294."
2,no_error_rows,True,Error rows: 0; warning rows: 0.
3,unique_metric_keys,True,Rows participating in duplicate metric keys: 0.
4,region_counts,True,All evaluation-region counts match expectations.
5,positive_region_pixel_counts,True,Successful rows with non-positive region size: 0.


,check_name,observed,expected,passed,failure_message
0,batch3_batch2_validation_passed,True,True,True,
1,batch3_input_rows,945,945,True,
2,batch3_zero_control_rows,94,94,True,
3,batch3_nonzero_rows,851,851,True,
4,batch3_input_paths_exist,"{'clean_path': 0, 'damaged_path': 0, 'mask_pat...",all zero,True,
5,batch3_expected_region_counts,"{'full_image': 945, 'content_region': 945, 'ma...","{'full_image': 945, 'content_region': 945, 'ma...",True,
6,batch3_expected_metric_rows,5294,5294,True,
7,batch3_metric_rows_match_expected,5294,5294,True,
8,batch3_metric_status_ok,{'ok': 5294},{'ok': 5294},True,
9,batch3_required_output_columns_present,[],[],True,


In [16]:
# Batch 4 / Cell 15 — Load Batch 3 outputs
import re
import textwrap

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

BATCH4_SUMMARY_PATH = OUTPUT_DIRS["analysis"] / "stable_diffusion_classical_metrics_summary.csv"
BATCH4_SELECTED_CASES_PATH = OUTPUT_DIRS["analysis"] / "stable_diffusion_classical_metric_selected_cases.csv"
BATCH4_FIGURE_DIR = OUTPUT_DIRS["figures"] / "stable_diffusion_classical_metric_diagnostics"
BATCH4_FIGURE_MANIFEST_PATH = OUTPUT_DIRS["figures"] / "stable_diffusion_classical_metric_figure_manifest.csv"
BATCH4_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch4_analysis_validation.csv"

BATCH4_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if "BATCH3_METRICS_PATH" not in globals():
    BATCH3_METRICS_PATH = OUTPUT_DIRS["metrics"] / "stable_diffusion_classical_metrics.csv"

if "BATCH3_VALIDATION_PATH" not in globals():
    BATCH3_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "stable_diffusion_classical_metrics_validation.csv"

if not BATCH3_METRICS_PATH.is_file():
    raise FileNotFoundError(f"Batch 3 metrics output is missing: {BATCH3_METRICS_PATH}")

if not BATCH3_VALIDATION_PATH.is_file():
    raise FileNotFoundError(f"Batch 3 validation output is missing: {BATCH3_VALIDATION_PATH}")

stable_diffusion_classical_metrics_df = pd.read_csv(BATCH3_METRICS_PATH)
batch3_validation_df = pd.read_csv(BATCH3_VALIDATION_PATH)
batch3_passed = bool(batch3_validation_df["passed"].astype(bool).all())

if STAGE_MANIFEST_PATH.is_file():
    stage_manifest = json.loads(STAGE_MANIFEST_PATH.read_text(encoding="utf-8"))

print(f"Batch 3 validation passed: {batch3_passed}")
print(f"Loaded Batch 3 metric rows: {len(stable_diffusion_classical_metrics_df):,}")
print(stable_diffusion_classical_metrics_df["evaluation_region"].value_counts().to_string())

display(stable_diffusion_classical_metrics_df.head(5))

Batch 3 validation passed: True
Loaded Batch 3 metric rows: 5,294
evaluation_region
full_image             945
content_region         945
masked_region          851
boundary_region        851
outside_mask_region    851
mask_bbox_crop         851


,dataset_name,case_id,painting_id,category,title,mask_id,mask_type,model_name,restoration_case_id,candidate_id,...,ssim_improvement,metric_module,metric_version,metric_timestamp_utc,python_version,numpy_version,pandas_version,skimage_version,status,issue
0,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,0.068244,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,NaN
1,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,0.079151,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,NaN
2,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,NaN
3,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,NaN
4,canonical,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,stable_diffusion_inpainting,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,...,NaN,restoration_eval.metrics_classical,2.1.0,2026-08-07T18:07:49.191686+00:00,3.12.6,1.26.4,2.3.3,0.24.0,ok,NaN


In [17]:
# Batch 4 / Cell 16 — Build consolidated summary table
if not batch3_passed:
    raise RuntimeError("Batch 3 validation did not pass. Fix Batch 3 before running Batch 4.")

metrics_ok_df = stable_diffusion_classical_metrics_df.loc[
    stable_diffusion_classical_metrics_df["status"].astype(str).eq("ok")
].copy()

masked_region_metrics_df = metrics_ok_df.loc[
    metrics_ok_df["evaluation_region"].astype(str).eq("masked_region")
].copy()

outside_mask_metrics_df = metrics_ok_df.loc[
    metrics_ok_df["evaluation_region"].astype(str).eq("outside_mask_region")
].copy()

zero_control_full_metrics_df = metrics_ok_df.loc[
    metrics_ok_df["mask_type"].astype(str).eq(ZERO_CONTROL_MASK_TYPE)
    & metrics_ok_df["evaluation_region"].astype(str).eq("full_image")
].copy()


def build_summary_scope(source_df: pd.DataFrame, scope_name: str, group_columns: list[str]) -> pd.DataFrame:
    available_group_columns = [column for column in group_columns if column in source_df.columns]
    if source_df.empty or len(available_group_columns) != len(group_columns):
        return pd.DataFrame()

    summary_df = metrics_classical.summarize_classical_metrics(source_df, available_group_columns)
    summary_df.insert(0, "summary_scope", scope_name)
    summary_df["summary_built_at_utc"] = utc_now_iso()
    summary_df["metric_version"] = metrics_classical.METRIC_VERSION
    return summary_df


overall_source_df = metrics_ok_df.copy()
overall_source_df["summary_group"] = "all_candidates"

summary_frames = [
    build_summary_scope(overall_source_df, "overall", ["summary_group"]),
    build_summary_scope(metrics_ok_df, "by_evaluation_region", ["evaluation_region"]),
    build_summary_scope(metrics_ok_df, "by_mask_type", ["mask_type"]),
    build_summary_scope(metrics_ok_df, "by_category", ["category"]),
    build_summary_scope(metrics_ok_df, "by_prompt_policy", ["prompt_policy_id"]),
    build_summary_scope(metrics_ok_df, "by_prompt_variant", ["prompt_variant_id"]),
    build_summary_scope(metrics_ok_df, "by_region_and_mask_type", ["evaluation_region", "mask_type"]),
    build_summary_scope(masked_region_metrics_df, "masked_region_by_mask_type", ["mask_type"]),
    build_summary_scope(masked_region_metrics_df, "masked_region_by_category", ["category"]),
    build_summary_scope(masked_region_metrics_df, "masked_region_by_prompt_variant", ["prompt_variant_id"]),
]

stable_diffusion_classical_metrics_summary_df = pd.concat(
    [frame for frame in summary_frames if not frame.empty],
    ignore_index=True,
)

summary_front_columns = [
    "summary_scope",
    "summary_group",
    "evaluation_region",
    "mask_type",
    "category",
    "prompt_policy_id",
    "prompt_variant_id",
    "rows",
    "cases",
    "mean_mse_improvement",
    "mean_mae_improvement",
    "mean_psnr_improvement",
    "mean_ssim_improvement",
]

summary_ordered_columns = [
    column for column in summary_front_columns
    if column in stable_diffusion_classical_metrics_summary_df.columns
] + [
    column for column in stable_diffusion_classical_metrics_summary_df.columns
    if column not in summary_front_columns
]

stable_diffusion_classical_metrics_summary_df = stable_diffusion_classical_metrics_summary_df[
    summary_ordered_columns
].sort_values(
    [column for column in ["summary_scope", "evaluation_region", "mask_type", "category", "prompt_variant_id"] if column in stable_diffusion_classical_metrics_summary_df.columns],
    kind="stable",
).reset_index(drop=True)

stable_diffusion_classical_metrics_summary_df.to_csv(BATCH4_SUMMARY_PATH, index=False)

print(f"Saved: {rel(BATCH4_SUMMARY_PATH)}")
print(f"Summary rows: {len(stable_diffusion_classical_metrics_summary_df):,}")
display(stable_diffusion_classical_metrics_summary_df.head(20))

Saved: outputs/22_stable_diffusion_classical_metrics/analysis/stable_diffusion_classical_metrics_summary.csv
Summary rows: 135


,summary_scope,summary_group,evaluation_region,mask_type,category,prompt_policy_id,prompt_variant_id,rows,cases,mean_mse_improvement,...,mean_restored_mse,mean_damaged_mae,mean_restored_mae,mean_damaged_psnr,mean_restored_psnr,mean_damaged_ssim,mean_restored_ssim,mean_region_pixel_count,summary_built_at_utc,metric_version
0,by_category,NaN,NaN,NaN,abstraction_surrealism,NaN,NaN,1094,93,3355.9283,...,598.2288,25.8384,8.8139,20.9537,23.8885,0.9366,0.8964,317524.4122,2026-08-07T18:39:56.945854+00:00,2.1.0
1,by_category,NaN,NaN,NaN,architecture_structured,NaN,NaN,1062,93,4351.7936,...,330.2285,28.6358,6.3528,21.5850,26.7845,0.9330,0.9038,293543.8117,2026-08-07T18:39:56.945854+00:00,2.1.0
2,by_category,NaN,NaN,NaN,high_texture_brushwork,NaN,NaN,1062,93,5231.4146,...,404.8611,32.5557,7.0909,19.8678,25.3671,0.9229,0.8972,316928.9623,2026-08-07T18:39:56.945854+00:00,2.1.0
3,by_category,NaN,NaN,NaN,landscape_natural,NaN,NaN,1038,93,4697.9007,...,384.3923,29.8677,6.5869,21.0611,26.1958,0.9280,0.9133,281958.9441,2026-08-07T18:39:56.945854+00:00,2.1.0
4,by_category,NaN,NaN,NaN,portrait_figure,NaN,NaN,1038,93,7474.1237,...,533.1415,39.1211,7.3048,20.7922,26.3911,0.9133,0.9165,324962.2736,2026-08-07T18:39:56.945854+00:00,2.1.0
5,by_evaluation_region,NaN,boundary_region,NaN,NaN,NaN,NaN,851,415,8119.9127,...,406.1579,47.5461,7.1311,19.9316,28.4952,NaN,NaN,13617.9718,2026-08-07T18:39:56.881864+00:00,2.1.0
6,by_evaluation_region,NaN,content_region,NaN,NaN,NaN,NaN,945,465,976.5094,...,226.8951,7.6381,3.6542,23.7369,27.7329,0.9492,0.9251,447008.5079,2026-08-07T18:39:56.881864+00:00,2.1.0
7,by_evaluation_region,NaN,full_image,NaN,NaN,NaN,NaN,945,465,744.2040,...,173.6084,5.7920,2.7676,24.9796,28.9756,0.9610,0.9436,589824.0000,2026-08-07T18:39:56.881864+00:00,2.1.0
8,by_evaluation_region,NaN,mask_bbox_crop,NaN,NaN,NaN,NaN,851,415,3536.3003,...,461.5283,23.7961,7.4438,21.2485,25.2445,0.8641,0.8408,299372.4877,2026-08-07T18:39:56.881864+00:00,2.1.0
9,by_evaluation_region,NaN,masked_region,NaN,NaN,NaN,NaN,851,415,17553.2344,...,1493.1265,107.4727,23.3460,14.3022,18.2982,NaN,NaN,101847.7650,2026-08-07T18:39:56.881864+00:00,2.1.0


In [18]:
# Batch 4 / Cell 17 — Build in-notebook rankings and selected diagnostic cases
ranking_display_columns = [
    column for column in [
        "candidate_id",
        "restoration_case_id",
        "case_id",
        "painting_id",
        "category",
        "mask_type",
        "prompt_policy_id",
        "prompt_variant_id",
        "candidate_seed",
        "evaluation_region",
        "mse_improvement",
        "mae_improvement",
        "psnr_improvement",
        "ssim_improvement",
        "restored_mse",
        "restored_ssim",
    ]
    if column in metrics_ok_df.columns
]

best_masked_mse_df = masked_region_metrics_df.sort_values(
    ["mse_improvement", "ssim_improvement"],
    ascending=[False, False],
    kind="stable",
).head(10)

weakest_masked_mse_df = masked_region_metrics_df.sort_values(
    ["mse_improvement", "ssim_improvement"],
    ascending=[True, True],
    kind="stable",
).head(10)

best_masked_ssim_df = masked_region_metrics_df.sort_values(
    ["ssim_improvement", "mse_improvement"],
    ascending=[False, False],
    kind="stable",
).head(10)

outside_mask_degradation_df = outside_mask_metrics_df.sort_values(
    ["mse_improvement", "mae_improvement"],
    ascending=[True, True],
    kind="stable",
).head(10)

zero_control_shift_df = zero_control_full_metrics_df.sort_values(
    ["restored_mse", "restored_mae"],
    ascending=[False, False],
    kind="stable",
).head(10)

print("Best masked-region MSE improvement")
display(best_masked_mse_df[ranking_display_columns])

print("Weakest masked-region MSE improvement")
display(weakest_masked_mse_df[ranking_display_columns])

print("Best masked-region SSIM improvement")
display(best_masked_ssim_df[ranking_display_columns])

print("Outside-mask degradation watch")
display(outside_mask_degradation_df[ranking_display_columns])

print("Zero-control largest restored shift")
display(zero_control_shift_df[ranking_display_columns])


def selection_frame(source_df: pd.DataFrame, reason: str, priority: int, n: int = 2) -> pd.DataFrame:
    selected_df = source_df.head(n).copy()
    if selected_df.empty:
        return selected_df
    selected_df["selection_reason"] = reason
    selected_df["selection_priority"] = priority
    selected_df["selection_rank"] = range(1, len(selected_df) + 1)
    return selected_df


category_representative_df = (
    masked_region_metrics_df.sort_values(
        ["category", "mse_improvement", "ssim_improvement"],
        ascending=[True, False, False],
        kind="stable",
    )
    .drop_duplicates("category", keep="first")
    .head(5)
)

raw_selected_cases_df = pd.concat(
    [
        selection_frame(best_masked_mse_df, "best_masked_mse_improvement", 10, n=2),
        selection_frame(weakest_masked_mse_df, "weakest_masked_mse_improvement", 20, n=2),
        selection_frame(best_masked_ssim_df, "best_masked_ssim_improvement", 30, n=2),
        selection_frame(outside_mask_degradation_df, "outside_mask_degradation_watch", 40, n=2),
        selection_frame(zero_control_shift_df, "zero_control_largest_restored_shift", 50, n=2),
        selection_frame(category_representative_df, "category_representative_best_masked_mse", 60, n=5),
    ],
    ignore_index=True,
)

if raw_selected_cases_df.empty:
    raise RuntimeError("No selected cases were produced for Batch 4 diagnostics.")

selected_reason_map = (
    raw_selected_cases_df
    .groupby("candidate_id", dropna=False)["selection_reason"]
    .agg(lambda values: " | ".join(dict.fromkeys(values.astype(str))))
    .rename("selection_reasons")
    .reset_index()
)

selected_cases_df = (
    raw_selected_cases_df
    .sort_values(["selection_priority", "selection_rank", "candidate_id"], kind="stable")
    .drop_duplicates("candidate_id", keep="first")
    .merge(selected_reason_map, on="candidate_id", how="left")
    .sort_values(["selection_priority", "selection_rank", "candidate_id"], kind="stable")
    .reset_index(drop=True)
)

selected_cases_df.insert(0, "selected_case_index", range(1, len(selected_cases_df) + 1))
selected_cases_df["selected_at_utc"] = utc_now_iso()

selected_front_columns = [
    "selected_case_index",
    "selection_reason",
    "selection_reasons",
    "selection_priority",
    "selection_rank",
    "candidate_id",
    "restoration_case_id",
    "case_id",
    "painting_id",
    "category",
    "title",
    "mask_type",
    "prompt_policy_id",
    "prompt_variant_id",
    "prompt_template_name",
    "candidate_seed",
    "evaluation_region",
    "mse_improvement",
    "mae_improvement",
    "psnr_improvement",
    "ssim_improvement",
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
]

selected_ordered_columns = [
    column for column in selected_front_columns
    if column in selected_cases_df.columns
] + [
    column for column in selected_cases_df.columns
    if column not in selected_front_columns
]

selected_cases_df = selected_cases_df[selected_ordered_columns]

print(f"Selected unique diagnostic cases: {len(selected_cases_df):,}")
display(selected_cases_df[selected_ordered_columns[: min(len(selected_ordered_columns), 24)]])

Best masked-region MSE improvement


,candidate_id,restoration_case_id,case_id,painting_id,category,mask_type,prompt_policy_id,prompt_variant_id,candidate_seed,evaluation_region,mse_improvement,mae_improvement,psnr_improvement,ssim_improvement,restored_mse,restored_ssim
8,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,2026,masked_region,54261.920673,227.298163,29.406277,NaN,62.282452,NaN
2770,sd__mrob__p001__loss_small__p00_generic__s2026...,sd__mrob__p001__loss_small__p00_generic__s2026...,p001__loss_small__variant_03,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,2026,masked_region,52572.774078,220.491221,25.044961,NaN,165.054047,NaN
2794,sd__mrob__p001__loss_small__p02_artist__s2026_...,sd__mrob__p001__loss_small__p02_artist__s2026_...,p001__loss_small__variant_05,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p02_artist,2026,masked_region,50613.847885,217.117750,25.948515,NaN,128.980240,NaN
2800,sd__mrob__p001__loss_small__p03_artist_style_p...,sd__mrob__p001__loss_small__p03_artist_style_p...,p001__loss_small__variant_05,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p03_artist_style_period,2026,masked_region,50609.214828,217.077735,25.795250,NaN,133.613297,NaN
2806,sd__mrob__p001__loss_small__p04_full_context__...,sd__mrob__p001__loss_small__p04_full_context__...,p001__loss_small__variant_05,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p04_full_context,2026,masked_region,50606.154419,217.042262,25.696897,NaN,136.673706,NaN
2788,sd__mrob__p001__loss_small__p01_style_period__...,sd__mrob__p001__loss_small__p01_style_period__...,p001__loss_small__variant_05,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p01_style_period,2026,masked_region,50598.432922,216.923601,25.458219,NaN,144.395203,NaN
2782,sd__mrob__p001__loss_small__p00_generic__s2026...,sd__mrob__p001__loss_small__p00_generic__s2026...,p001__loss_small__variant_05,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,2026,masked_region,50588.073746,216.768288,25.157317,NaN,154.754379,NaN
2776,sd__mrob__p001__loss_small__p00_generic__s2026...,sd__mrob__p001__loss_small__p00_generic__s2026...,p001__loss_small__variant_04,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,2026,masked_region,50477.951172,217.170151,25.327365,NaN,148.470703,NaN
2758,sd__mrob__p001__loss_small__p00_generic__s2026...,sd__mrob__p001__loss_small__p00_generic__s2026...,p001__loss_small__variant_01,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,2026,masked_region,50282.820404,210.115296,20.228577,NaN,481.617096,NaN
14,sd__can__p001__mixed_damage__p00_generic__s202...,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,p001,portrait_figure,mixed_damage,sd_inpaint_prompt_ablation_v1,p00_generic,2026,masked_region,50101.041138,208.660862,18.026954,NaN,801.763550,NaN


Weakest masked-region MSE improvement


,candidate_id,restoration_case_id,case_id,painting_id,category,mask_type,prompt_policy_id,prompt_variant_id,candidate_seed,evaluation_region,mse_improvement,mae_improvement,psnr_improvement,ssim_improvement,restored_mse,restored_ssim
1690,sd__can__p036__scratch_thin__p00_generic__s202...,sd__can__p036__scratch_thin__p00_generic__s202...,p036_scratch_thin,p036,abstraction_surrealism,scratch_thin,sd_inpaint_prompt_ablation_v1,p00_generic,2026,masked_region,-4040.078125,-38.987049,-3.058164,NaN,7992.553711,NaN
4918,sd__syn__p039__water_stain__p04_full_context__...,sd__syn__p039__water_stain__p04_full_context__...,p039__water_stain__severe,p039,abstraction_surrealism,water_stain,sd_inpaint_prompt_ablation_v1,p04_full_context,2026,masked_region,-3017.133186,-33.737735,-15.084299,NaN,3113.704346,NaN
4906,sd__syn__p039__water_stain__p02_artist__s2026_...,sd__syn__p039__water_stain__p02_artist__s2026_...,p039__water_stain__severe,p039,abstraction_surrealism,water_stain,sd_inpaint_prompt_ablation_v1,p02_artist,2026,masked_region,-2544.350471,-29.673694,-14.369081,NaN,2640.921631,NaN
4912,sd__syn__p039__water_stain__p03_artist_style_p...,sd__syn__p039__water_stain__p03_artist_style_p...,p039__water_stain__severe,p039,abstraction_surrealism,water_stain,sd_inpaint_prompt_ablation_v1,p03_artist_style_period,2026,masked_region,-2434.915657,-28.780765,-14.185282,NaN,2531.486816,NaN
3820,sd__syn__p001__partial_transparency__p03_artis...,sd__syn__p001__partial_transparency__p03_artis...,p001__partial_transparency__severe,p001,portrait_figure,partial_transparency,sd_inpaint_prompt_ablation_v1,p03_artist_style_period,2026,masked_region,-2411.856445,-11.561941,-4.536319,NaN,3721.188477,NaN
3814,sd__syn__p001__partial_transparency__p02_artis...,sd__syn__p001__partial_transparency__p02_artis...,p001__partial_transparency__severe,p001,portrait_figure,partial_transparency,sd_inpaint_prompt_ablation_v1,p02_artist,2026,masked_region,-2293.698730,-10.619078,-4.396182,NaN,3603.030762,NaN
4894,sd__syn__p039__water_stain__p00_generic__s2026...,sd__syn__p039__water_stain__p00_generic__s2026...,p039__water_stain__severe,p039,abstraction_surrealism,water_stain,sd_inpaint_prompt_ablation_v1,p00_generic,2026,masked_region,-2292.577522,-26.733016,-13.933957,NaN,2389.148682,NaN
3574,sd__syn__p001__blur__p00_generic__s2026__a0008...,sd__syn__p001__blur__p00_generic__s2026__a0008...,p001__blur__severe,p001,portrait_figure,blur,sd_inpaint_prompt_ablation_v1,p00_generic,2026,masked_region,-2234.206309,-24.354511,-21.998703,NaN,2248.396973,NaN
3808,sd__syn__p001__partial_transparency__p01_style...,sd__syn__p001__partial_transparency__p01_style...,p001__partial_transparency__severe,p001,portrait_figure,partial_transparency,sd_inpaint_prompt_ablation_v1,p01_style_period,2026,masked_region,-2211.261475,-11.825949,-4.295661,NaN,3520.593506,NaN
5230,sd__syn__p043__water_stain__p00_generic__s2026...,sd__syn__p043__water_stain__p00_generic__s2026...,p043__water_stain__moderate,p043,high_texture_brushwork,water_stain,sd_inpaint_prompt_ablation_v1,p00_generic,2026,masked_region,-2196.664803,-27.709780,-17.818406,NaN,2233.576172,NaN


Best masked-region SSIM improvement


...

,candidate_id,restoration_case_id,case_id,painting_id,category,mask_type,prompt_policy_id,prompt_variant_id,candidate_seed,evaluation_region,mse_improvement,mae_improvement,psnr_improvement,ssim_improvement,restored_mse,restored_ssim
4,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001,portrait_figure,loss_large,sd_inpaint_prompt_ablation_v1,p00_generic,2026,outside_mask_region,0.0,0.0,0.0,NaN,0.0,NaN
10,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,2026,outside_mask_region,0.0,0.0,0.0,NaN,0.0,NaN
16,sd__can__p001__mixed_damage__p00_generic__s202...,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,p001,portrait_figure,mixed_damage,sd_inpaint_prompt_ablation_v1,p00_generic,2026,outside_mask_region,0.0,0.0,0.0,NaN,0.0,NaN
22,sd__can__p001__scratch_thin__p00_generic__s202...,sd__can__p001__scratch_thin__p00_generic__s202...,p001_scratch_thin,p001,portrait_figure,scratch_thin,sd_inpaint_prompt_ablation_v1,p00_generic,2026,outside_mask_region,0.0,0.0,0.0,NaN,0.0,NaN
30,sd__can__p002__loss_large__p00_generic__s2026_...,sd__can__p002__loss_large__p00_generic__s2026_...,p002_loss_large,p002,portrait_figure,loss_large,sd_inpaint_prompt_ablation_v1,p00_generic,2026,outside_mask_region,0.0,0.0,0.0,NaN,0.0,NaN
36,sd__can__p002__loss_small__p00_generic__s2026_...,sd__can__p002__loss_small__p00_generic__s2026_...,p002_loss_small,p002,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,2026,outside_mask_region,0.0,0.0,0.0,NaN,0.0,NaN
42,sd__can__p002__mixed_damage__p00_generic__s202...,sd__can__p002__mixed_damage__p00_generic__s202...,p002_mixed_damage,p002,portrait_figure,mixed_damage,sd_inpaint_prompt_ablation_v1,p00_generic,2026,outside_mask_region,0.0,0.0,0.0,NaN,0.0,NaN
48,sd__can__p002__scratch_thin__p00_generic__s202...,sd__can__p002__scratch_thin__p00_generic__s202...,p002_scratch_thin,p002,portrait_figure,scratch_thin,sd_inpaint_prompt_ablation_v1,p00_generic,2026,outside_mask_region,0.0,0.0,0.0,NaN,0.0,NaN
56,sd__can__p003__loss_large__p00_generic__s2026_...,sd__can__p003__loss_large__p00_generic__s2026_...,p003_loss_large,p003,portrait_figure,loss_large,sd_inpaint_prompt_ablation_v1,p00_generic,2026,outside_mask_region,0.0,0.0,0.0,NaN,0.0,NaN
62,sd__can__p003__loss_small__p00_generic__s2026_...,sd__can__p003__loss_small__p00_generic__s2026_...,p003_loss_small,p003,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,2026,outside_mask_region,0.0,0.0,0.0,NaN,0.0,NaN


Zero-control largest restored shift


,candidate_id,restoration_case_id,case_id,painting_id,category,mask_type,prompt_policy_id,prompt_variant_id,candidate_seed,evaluation_region,mse_improvement,mae_improvement,psnr_improvement,ssim_improvement,restored_mse,restored_ssim
24,sd__can__p001__zero__p00_generic__s2026__3be10...,sd__can__p001__zero__p00_generic__s2026__3be10...,p001_zero_control,p001,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p00_generic,2026,full_image,0.0,0.0,0.0,0.0,0.0,1.0
50,sd__can__p002__zero__p00_generic__s2026__606dc...,sd__can__p002__zero__p00_generic__s2026__606dc...,p002_zero_control,p002,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p00_generic,2026,full_image,0.0,0.0,0.0,0.0,0.0,1.0
76,sd__can__p003__zero__p00_generic__s2026__b541c...,sd__can__p003__zero__p00_generic__s2026__b541c...,p003_zero_control,p003,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p00_generic,2026,full_image,0.0,0.0,0.0,0.0,0.0,1.0
198,sd__can__p004__zero__p00_generic__s2026__f5472...,sd__can__p004__zero__p00_generic__s2026__f5472...,p004_zero_control,p004,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p00_generic,2026,full_image,0.0,0.0,0.0,0.0,0.0,1.0
200,sd__can__p004__zero__p01_style_period__s2026__...,sd__can__p004__zero__p01_style_period__s2026__...,p004_zero_control,p004,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p01_style_period,2026,full_image,0.0,0.0,0.0,0.0,0.0,1.0
202,sd__can__p004__zero__p02_artist__s2026__ff4143...,sd__can__p004__zero__p02_artist__s2026__ff4143...,p004_zero_control,p004,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p02_artist,2026,full_image,0.0,0.0,0.0,0.0,0.0,1.0
204,sd__can__p004__zero__p03_artist_style_p__s2026...,sd__can__p004__zero__p03_artist_style_p__s2026...,p004_zero_control,p004,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p03_artist_style_period,2026,full_image,0.0,0.0,0.0,0.0,0.0,1.0
206,sd__can__p004__zero__p04_full_context__s2026__...,sd__can__p004__zero__p04_full_context__s2026__...,p004_zero_control,p004,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p04_full_context,2026,full_image,0.0,0.0,0.0,0.0,0.0,1.0
232,sd__can__p005__zero__p00_generic__s2026__721e5...,sd__can__p005__zero__p00_generic__s2026__721e5...,p005_zero_control,p005,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p00_generic,2026,full_image,0.0,0.0,0.0,0.0,0.0,1.0
258,sd__can__p006__zero__p00_generic__s2026__673d8...,sd__can__p006__zero__p00_generic__s2026__673d8...,p006_zero_control,p006,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p00_generic,2026,full_image,0.0,0.0,0.0,0.0,0.0,1.0


Selected unique diagnostic cases: 11


,selected_case_index,selection_reason,selection_reasons,selection_priority,selection_rank,candidate_id,restoration_case_id,case_id,painting_id,category,...,prompt_template_name,candidate_seed,evaluation_region,mse_improvement,mae_improvement,psnr_improvement,ssim_improvement,clean_path,damaged_path,mask_path
0,1,best_masked_mse_improvement,best_masked_mse_improvement | best_masked_ssim...,10,1,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001,portrait_figure,...,generic_restoration,2026,masked_region,54261.920673,227.298163,29.406277,NaN,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_small_damaged.png,data/processed/masks/p001_loss_small_mask.png
1,2,best_masked_mse_improvement,best_masked_mse_improvement | best_masked_ssim...,10,2,sd__mrob__p001__loss_small__p00_generic__s2026...,sd__mrob__p001__loss_small__p00_generic__s2026...,p001__loss_small__variant_03,p001,portrait_figure,...,generic_restoration,2026,masked_region,52572.774078,220.491221,25.044961,NaN,data/processed/clean/p001_clean.png,data/processed/damaged/mask_robustness/p001__l...,data/processed/masks/mask_robustness/p001__los...
2,3,weakest_masked_mse_improvement,weakest_masked_mse_improvement,20,1,sd__can__p036__scratch_thin__p00_generic__s202...,sd__can__p036__scratch_thin__p00_generic__s202...,p036_scratch_thin,p036,abstraction_surrealism,...,generic_restoration,2026,masked_region,-4040.078125,-38.987049,-3.058164,NaN,data/processed/clean/p036_clean.png,data/processed/masked/p036_scratch_thin_damage...,data/processed/masks/p036_scratch_thin_mask.png
3,4,weakest_masked_mse_improvement,weakest_masked_mse_improvement,20,2,sd__syn__p039__water_stain__p04_full_context__...,sd__syn__p039__water_stain__p04_full_context__...,p039__water_stain__severe,p039,abstraction_surrealism,...,artwork_full_context,2026,masked_region,-3017.133186,-33.737735,-15.084299,NaN,data/processed/clean/p039_clean.png,data/processed/degraded/synthetic_degradation/...,data/processed/masks/synthetic_degradation/p03...
4,5,outside_mask_degradation_watch,outside_mask_degradation_watch,40,1,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001,portrait_figure,...,generic_restoration,2026,outside_mask_region,0.000000,0.000000,0.000000,NaN,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_large_damaged.png,data/processed/masks/p001_loss_large_mask.png
5,6,zero_control_largest_restored_shift,zero_control_largest_restored_shift,50,1,sd__can__p001__zero__p00_generic__s2026__3be10...,sd__can__p001__zero__p00_generic__s2026__3be10...,p001_zero_control,p001,portrait_figure,...,generic_restoration,2026,full_image,0.000000,0.000000,0.000000,0.0,data/processed/clean/p001_clean.png,data/processed/masked/p001_zero_control_damage...,data/processed/masks/p001_zero_control_mask.png
6,7,zero_control_largest_restored_shift,zero_control_largest_restored_shift,50,2,sd__can__p002__zero__p00_generic__s2026__606dc...,sd__can__p002__zero__p00_generic__s2026__606dc...,p002_zero_control,p002,portrait_figure,...,generic_restoration,2026,full_image,0.000000,0.000000,0.000000,0.0,data/processed/clean/p002_clean.png,data/processed/masked/p002_zero_control_damage...,data/processed/masks/p002_zero_control_mask.png
7,8,category_representative_best_masked_mse,category_representative_best_masked_mse,60,1,sd__can__p040__loss_large__p00_generic__s2026_...,sd__can__p040__loss_large__p00_generic__s2026_...,p040_loss_large,p040,abstraction_surrealism,...,generic_restoration,2026,masked_region,30905.328613,133.857136,8.758928,NaN,data/processed/clean/p040_clean.png,data/processed/masked/p040_loss_large_damaged.png,data/processed/masks/p040_loss_large_mask.png
8,9,category_representative_best_masked_mse,category_representative_best_masked_mse,60,2,sd__can__p024__loss_small__p00_generic__s2026_...,sd__can__p024__loss_small__p00_generic__s2026_...,p024_loss_small,p02

In [19]:
# Batch 4 / Cell 18 — Generate diagnostic figures with fixed short filenames
def resolve_project_file(path_value) -> Path:
    if pd.isna(path_value) or str(path_value).strip() == "":
        raise ValueError("Missing path value.")
    candidate_path = Path(str(path_value))
    if candidate_path.is_absolute():
        return candidate_path
    return PROJECT_ROOT / candidate_path


def load_rgb_for_display(path_value) -> Image.Image:
    image_path = resolve_project_file(path_value)
    if not image_path.is_file():
        raise FileNotFoundError(f"Missing image file: {image_path}")
    return Image.open(image_path).convert("RGB")


def load_mask_for_display(path_value, size: tuple[int, int]) -> Image.Image:
    mask_path = resolve_project_file(path_value)
    if not mask_path.is_file():
        raise FileNotFoundError(f"Missing mask file: {mask_path}")

    mask_img = Image.open(mask_path).convert("L")
    if mask_img.size != size:
        resampling = getattr(Image, "Resampling", Image).NEAREST
        mask_img = mask_img.resize(size, resampling)
    return mask_img


def resize_to_match(image: Image.Image, size: tuple[int, int]) -> Image.Image:
    if image.size == size:
        return image
    resampling = getattr(Image, "Resampling", Image).BILINEAR
    return image.resize(size, resampling)


def safe_title(value, width: int = 42) -> str:
    return textwrap.shorten(str(value), width=width, placeholder="...")


def save_batch4_diagnostic_figure(row: pd.Series, figure_path: Path) -> None:
    clean_img = load_rgb_for_display(row["clean_path"])
    damaged_img = resize_to_match(load_rgb_for_display(row["damaged_path"]), clean_img.size)
    restored_img = resize_to_match(load_rgb_for_display(row["restored_path"]), clean_img.size)
    mask_img = load_mask_for_display(row["mask_path"], clean_img.size)

    clean_arr = np.asarray(clean_img).astype(np.float32)
    restored_arr = np.asarray(restored_img).astype(np.float32)
    abs_diff = np.abs(restored_arr - clean_arr).mean(axis=2)

    fig, axes = plt.subplots(1, 5, figsize=(17, 4))

    panels = [
        ("Clean", clean_img, None),
        ("Damaged", damaged_img, None),
        ("Mask", mask_img, "gray"),
        ("Restored", restored_img, None),
        ("|Restored - Clean|", abs_diff, "magma"),
    ]

    for axis, (title, image_value, cmap) in zip(axes, panels):
        axis.imshow(image_value, cmap=cmap)
        axis.set_title(title, fontsize=10)
        axis.axis("off")

    figure_title = (
        f"B4-{int(row['selected_case_index']):03d} | "
        f"{safe_title(row.get('selection_reason', 'selected'))} | "
        f"{safe_title(row.get('candidate_id', 'candidate'))}"
    )

    fig.suptitle(figure_title, fontsize=11)
    fig.tight_layout()
    fig.savefig(figure_path, dpi=140, bbox_inches="tight")
    plt.close(fig)


figure_manifest_rows = []

for _, selected_row in selected_cases_df.iterrows():
    selected_index = int(selected_row["selected_case_index"])
    figure_filename = f"b4_fig_{selected_index:03d}.png"
    figure_path = BATCH4_FIGURE_DIR / figure_filename

    save_batch4_diagnostic_figure(selected_row, figure_path)

    figure_manifest_rows.append(
        {
            "selected_case_index": selected_index,
            "figure_id": f"b4_fig_{selected_index:03d}",
            "figure_filename": figure_filename,
            "figure_path": rel(figure_path),
            "figure_filename_length": len(figure_filename),
            "figure_relative_path_length": len(rel(figure_path)),
            "candidate_id": selected_row.get("candidate_id", ""),
            "restoration_case_id": selected_row.get("restoration_case_id", ""),
            "case_id": selected_row.get("case_id", ""),
            "painting_id": selected_row.get("painting_id", ""),
            "category": selected_row.get("category", ""),
            "mask_type": selected_row.get("mask_type", ""),
            "prompt_policy_id": selected_row.get("prompt_policy_id", ""),
            "prompt_variant_id": selected_row.get("prompt_variant_id", ""),
            "selection_reason": selected_row.get("selection_reason", ""),
            "selection_reasons": selected_row.get("selection_reasons", ""),
            "mse_improvement": selected_row.get("mse_improvement", np.nan),
            "ssim_improvement": selected_row.get("ssim_improvement", np.nan),
            "created_at_utc": utc_now_iso(),
        }
    )

figure_manifest_df = pd.DataFrame(figure_manifest_rows)

selected_cases_df = selected_cases_df.merge(
    figure_manifest_df[["selected_case_index", "figure_path"]],
    on="selected_case_index",
    how="left",
)

selected_cases_df.to_csv(BATCH4_SELECTED_CASES_PATH, index=False)
figure_manifest_df.to_csv(BATCH4_FIGURE_MANIFEST_PATH, index=False)

print(f"Saved: {rel(BATCH4_SELECTED_CASES_PATH)}")
print(f"Saved: {rel(BATCH4_FIGURE_MANIFEST_PATH)}")
print(f"Generated diagnostic figures: {len(figure_manifest_df):,}")

display(figure_manifest_df.head(10))

Saved: outputs/22_stable_diffusion_classical_metrics/analysis/stable_diffusion_classical_metric_selected_cases.csv
Saved: outputs/22_stable_diffusion_classical_metrics/figures/stable_diffusion_classical_metric_figure_manifest.csv
Generated diagnostic figures: 11


,selected_case_index,figure_id,figure_filename,figure_path,figure_filename_length,figure_relative_path_length,candidate_id,restoration_case_id,case_id,painting_id,category,mask_type,prompt_policy_id,prompt_variant_id,selection_reason,selection_reasons,mse_improvement,ssim_improvement,created_at_utc
0,1,b4_fig_001,b4_fig_001.png,outputs/22_stable_diffusion_classical_metrics/...,14,114,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,best_masked_mse_improvement,best_masked_mse_improvement | best_masked_ssim...,54261.920673,NaN,2026-08-07T18:39:59.683667+00:00
1,2,b4_fig_002,b4_fig_002.png,outputs/22_stable_diffusion_classical_metrics/...,14,114,sd__mrob__p001__loss_small__p00_generic__s2026...,sd__mrob__p001__loss_small__p00_generic__s2026...,p001__loss_small__variant_03,p001,portrait_figure,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,best_masked_mse_improvement,best_masked_mse_improvement | best_masked_ssim...,52572.774078,NaN,2026-08-07T18:40:01.857960+00:00
2,3,b4_fig_003,b4_fig_003.png,outputs/22_stable_diffusion_classical_metrics/...,14,114,sd__can__p036__scratch_thin__p00_generic__s202...,sd__can__p036__scratch_thin__p00_generic__s202...,p036_scratch_thin,p036,abstraction_surrealism,scratch_thin,sd_inpaint_prompt_ablation_v1,p00_generic,weakest_masked_mse_improvement,weakest_masked_mse_improvement,-4040.078125,NaN,2026-08-07T18:40:03.691523+00:00
3,4,b4_fig_004,b4_fig_004.png,outputs/22_stable_diffusion_classical_metrics/...,14,114,sd__syn__p039__water_stain__p04_full_context__...,sd__syn__p039__water_stain__p04_full_context__...,p039__water_stain__severe,p039,abstraction_surrealism,water_stain,sd_inpaint_prompt_ablation_v1,p04_full_context,weakest_masked_mse_improvement,weakest_masked_mse_improvement,-3017.133186,NaN,2026-08-07T18:40:05.811359+00:00
4,5,b4_fig_005,b4_fig_005.png,outputs/22_stable_diffusion_classical_metrics/...,14,114,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001,portrait_figure,loss_large,sd_inpaint_prompt_ablation_v1,p00_generic,outside_mask_degradation_watch,outside_mask_degradation_watch,0.000000,NaN,2026-08-07T18:40:07.962576+00:00
5,6,b4_fig_006,b4_fig_006.png,outputs/22_stable_diffusion_classical_metrics/...,14,114,sd__can__p001__zero__p00_generic__s2026__3be10...,sd__can__p001__zero__p00_generic__s2026__3be10...,p001_zero_control,p001,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p00_generic,zero_control_largest_restored_shift,zero_control_largest_restored_shift,0.000000,0.0,2026-08-07T18:40:10.224075+00:00
6,7,b4_fig_007,b4_fig_007.png,outputs/22_stable_diffusion_classical_metrics/...,14,114,sd__can__p002__zero__p00_generic__s2026__606dc...,sd__can__p002__zero__p00_generic__s2026__606dc...,p002_zero_control,p002,portrait_figure,zero_control,sd_inpaint_prompt_ablation_v1,p00_generic,zero_control_largest_restored_shift,zero_control_largest_restored_shift,0.000000,0.0,2026-08-07T18:40:12.168305+00:00
7,8,b4_fig_008,b4_fig_008.png,outputs/22_stable_diffusion_classical_metrics/...,14,114,sd__can__p040__loss_large__p00_generic__s2026_...,sd__can__p040__loss_large__p00_generic__s2026_...,p040_loss_large,p040,abstraction_surrealism,loss_large,sd_inpaint_prompt_ablation_v1,p00_generic,category_representative_best_masked_mse,category_representative_best_masked_mse,30905.328613,NaN,2026-08-07T18:40:14.235663+00:00
8,9,b4_fig_009,b4_fig_009.png,outputs/22_stable_diffusion_classical_metrics/...,14,114,sd__can__p024__loss_small__p00_generic__s2026_...,sd__can__p024__loss_small__p00_generic__s2026_...,p024_loss_small,p024,architecture_structured,loss_small,sd_inpaint_prompt_ablation_v1,p00_generic,category_representative_best_masked_mse,category_representative_best_masked_mse,41754.214874,NaN,2026-08-07T18:40:16.293506+00:00
9,10,b4_fig_010,b4_fig_010.png,outputs/22_stable_diff

In [20]:
# Batch 4 / Cell 19 — Validate Batch 4 outputs and update stage manifest
figure_paths_exist = [
    (PROJECT_ROOT / path).is_file()
    for path in figure_manifest_df["figure_path"].astype(str)
]

figure_filename_pattern_ok = figure_manifest_df["figure_filename"].astype(str).map(
    lambda value: bool(re.fullmatch(r"b4_fig_\d{3}\.png", value))
)

expected_summary_scopes = {
    "overall",
    "by_evaluation_region",
    "by_mask_type",
    "by_region_and_mask_type",
    "masked_region_by_mask_type",
}

actual_summary_scopes = set(
    stable_diffusion_classical_metrics_summary_df["summary_scope"].astype(str)
)

selection_reason_text = " | ".join(selected_cases_df["selection_reasons"].fillna("").astype(str))

required_selection_reasons = [
    "best_masked_mse_improvement",
    "weakest_masked_mse_improvement",
    "best_masked_ssim_improvement",
    "outside_mask_degradation_watch",
    "zero_control_largest_restored_shift",
]

batch4_output_files = [
    BATCH4_SUMMARY_PATH,
    BATCH4_SELECTED_CASES_PATH,
    BATCH4_FIGURE_MANIFEST_PATH,
]

batch4_validation_rows = [
    validation_row(
        "batch4_batch3_validation_passed",
        batch3_passed,
        True,
        batch3_passed,
        "Batch 3 validation did not pass before Batch 4.",
    ),
    validation_row(
        "batch4_summary_written",
        rel(BATCH4_SUMMARY_PATH),
        "file exists",
        BATCH4_SUMMARY_PATH.is_file(),
        "Summary CSV was not written.",
    ),
    validation_row(
        "batch4_summary_not_empty",
        len(stable_diffusion_classical_metrics_summary_df),
        "> 0",
        len(stable_diffusion_classical_metrics_summary_df) > 0,
        "Summary table is empty.",
    ),
    validation_row(
        "batch4_required_summary_scopes_present",
        sorted(actual_summary_scopes),
        sorted(expected_summary_scopes),
        expected_summary_scopes.issubset(actual_summary_scopes),
        "One or more required summary scopes are missing.",
    ),
    validation_row(
        "batch4_selected_cases_written",
        rel(BATCH4_SELECTED_CASES_PATH),
        "file exists",
        BATCH4_SELECTED_CASES_PATH.is_file(),
        "Selected cases CSV was not written.",
    ),
    validation_row(
        "batch4_selected_cases_not_empty",
        len(selected_cases_df),
        "> 0",
        len(selected_cases_df) > 0,
        "Selected cases table is empty.",
    ),
    validation_row(
        "batch4_required_selection_reasons_present",
        required_selection_reasons,
        "all present in selection_reasons",
        all(reason in selection_reason_text for reason in required_selection_reasons),
        "One or more required diagnostic selection reasons are missing.",
    ),
    validation_row(
        "batch4_figure_manifest_written",
        rel(BATCH4_FIGURE_MANIFEST_PATH),
        "file exists",
        BATCH4_FIGURE_MANIFEST_PATH.is_file(),
        "Figure manifest CSV was not written.",
    ),
    validation_row(
        "batch4_figure_count_matches_selected_cases",
        len(figure_manifest_df),
        len(selected_cases_df),
        len(figure_manifest_df) == len(selected_cases_df),
        "Figure manifest row count does not match selected cases.",
    ),
    validation_row(
        "batch4_figure_files_exist",
        int(sum(figure_paths_exist)),
        len(figure_manifest_df),
        all(figure_paths_exist),
        "One or more diagnostic PNGs are missing.",
    ),
    validation_row(
        "batch4_fixed_short_figure_filenames",
        figure_manifest_df["figure_filename_length"].max(),
        "<= 20",
        int(figure_manifest_df["figure_filename_length"].max()) <= 20,
        "Diagnostic figure filenames are too long.",
    ),
    validation_row(
        "batch4_no_case_derived_figure_filenames",
        bool(figure_filename_pattern_ok.all()),
        True,
        bool(figure_filename_pattern_ok.all()),
        "One or more figure filenames are not fixed short b4_fig_### names.",
    ),
    validation_row(
        "batch4_short_relative_figure_paths",
        int(figure_manifest_df["figure_relative_path_length"].max()),
        "<= 180",
        int(figure_manifest_df["figure_relative_path_length"].max()) <= 180,
        "One or more figure relative paths are too long.",
    ),
    validation_row(
        "batch4_fixed_output_csv_paths_exist",
        {rel(path): path.is_file() for path in batch4_output_files},
        "all true",
        all(path.is_file() for path in batch4_output_files),
        "One or more Batch 4 fixed output CSVs are missing.",
    ),
]

batch4_validation_df = pd.DataFrame(batch4_validation_rows)
batch4_validation_df.to_csv(BATCH4_VALIDATION_PATH, index=False)

stage_manifest["batch4"] = {
    "completed_at_utc": utc_now_iso(),
    "summary_rows": int(len(stable_diffusion_classical_metrics_summary_df)),
    "summary_scopes": sorted(actual_summary_scopes),
    "selected_cases": int(len(selected_cases_df)),
    "diagnostic_figures": int(len(figure_manifest_df)),
    "filename_length_policy": {
        "uses_fixed_short_figure_filenames": True,
        "figure_filename_pattern": "b4_fig_###.png",
        "max_figure_filename_length": int(figure_manifest_df["figure_filename_length"].max()),
        "max_figure_relative_path_length": int(figure_manifest_df["figure_relative_path_length"].max()),
        "no_case_or_prompt_derived_figure_filenames": bool(figure_filename_pattern_ok.all()),
    },
    "outputs": {
        "classical_metrics_summary": rel(BATCH4_SUMMARY_PATH),
        "selected_cases": rel(BATCH4_SELECTED_CASES_PATH),
        "figure_manifest": rel(BATCH4_FIGURE_MANIFEST_PATH),
        "diagnostic_figure_dir": rel(BATCH4_FIGURE_DIR),
        "batch4_validation": rel(BATCH4_VALIDATION_PATH),
    },
}

STAGE_MANIFEST_PATH.write_text(json.dumps(stage_manifest, indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH4_VALIDATION_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 4 checks passed: {int(batch4_validation_df['passed'].sum())} / {len(batch4_validation_df)}")

display(batch4_validation_df)

if not batch4_validation_df["passed"].all():
    display(batch4_validation_df.loc[~batch4_validation_df["passed"], ["check_name", "failure_message"]])
    raise RuntimeError("Batch 4 validation failed. Fix summaries/selected cases/figures before Batch 5.")

Saved: outputs/22_stable_diffusion_classical_metrics/validation/batch4_analysis_validation.csv
Updated: outputs/22_stable_diffusion_classical_metrics/manifests/stable_diffusion_classical_metrics_stage_manifest.json
Batch 4 checks passed: 14 / 14


,check_name,observed,expected,passed,failure_message
0,batch4_batch3_validation_passed,True,True,True,
1,batch4_summary_written,outputs/22_stable_diffusion_classical_metrics/...,file exists,True,
2,batch4_summary_not_empty,135,> 0,True,
3,batch4_required_summary_scopes_present,"[by_category, by_evaluation_region, by_mask_ty...","[by_evaluation_region, by_mask_type, by_region...",True,
4,batch4_selected_cases_written,outputs/22_stable_diffusion_classical_metrics/...,file exists,True,
5,batch4_selected_cases_not_empty,11,> 0,True,
6,batch4_required_selection_reasons_present,"[best_masked_mse_improvement, weakest_masked_m...",all present in selection_reasons,True,
7,batch4_figure_manifest_written,outputs/22_stable_diffusion_classical_metrics/...,file exists,True,
8,batch4_figure_count_matches_selected_cases,11,11,True,
9,batch4_figure_files_exist,11,11,True,


In [24]:
import hashlib
from typing import Any

import numpy as np

BATCH5_FINAL_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "stable_diffusion_classical_metrics_final_validation.csv"

ARTIFACT_INDEX_PATH = globals().get(
    "ARTIFACT_INDEX_PATH",
    OUTPUT_DIRS["manifests"] / "stable_diffusion_classical_metrics_artifact_index.csv",
)
HANDOFF_MANIFEST_PATH = globals().get(
    "HANDOFF_MANIFEST_PATH",
    OUTPUT_DIRS["manifests"] / "stable_diffusion_classical_metrics_handoff_manifest.json",
)

BATCH1_INPUT_CASES_PATH = globals().get(
    "BATCH1_INPUT_CASES_PATH",
    OUTPUT_DIRS["tables"] / "stable_diffusion_classical_metric_input_cases.csv",
)
BATCH3_METRICS_PATH = globals().get(
    "BATCH3_METRICS_PATH",
    OUTPUT_DIRS["metrics"] / "stable_diffusion_classical_metrics.csv",
)
BATCH3_VALIDATION_PATH = globals().get(
    "BATCH3_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "stable_diffusion_classical_metrics_validation.csv",
)
BATCH4_SUMMARY_PATH = globals().get(
    "BATCH4_SUMMARY_PATH",
    OUTPUT_DIRS["analysis"] / "stable_diffusion_classical_metrics_summary.csv",
)
BATCH4_SELECTED_CASES_PATH = globals().get(
    "BATCH4_SELECTED_CASES_PATH",
    OUTPUT_DIRS["analysis"] / "stable_diffusion_classical_metric_selected_cases.csv",
)
BATCH4_FIGURE_MANIFEST_PATH = globals().get(
    "BATCH4_FIGURE_MANIFEST_PATH",
    OUTPUT_DIRS["figures"] / "stable_diffusion_classical_metric_figure_manifest.csv",
)
BATCH4_FIGURE_DIR = globals().get(
    "BATCH4_FIGURE_DIR",
    OUTPUT_DIRS["figures"] / "stable_diffusion_classical_metric_diagnostics",
)
BATCH4_VALIDATION_PATH = globals().get(
    "BATCH4_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "batch4_analysis_validation.csv",
)

EXPECTED_SD_CANDIDATE_ROWS = globals().get("EXPECTED_SD_CANDIDATE_ROWS", 945)
EXPECTED_SD_ZERO_CONTROL_ROWS = globals().get("EXPECTED_SD_ZERO_CONTROL_ROWS", 94)
EXPECTED_SD_NONZERO_ROWS = globals().get("EXPECTED_SD_NONZERO_ROWS", 851)
EXPECTED_SD_METRIC_ROWS = globals().get("EXPECTED_SD_METRIC_ROWS", 5294)
ZERO_CONTROL_MASK_TYPE = globals().get("ZERO_CONTROL_MASK_TYPE", "zero_control")

EXPECTED_BATCH5_REGION_COUNTS = {
    "full_image": EXPECTED_SD_CANDIDATE_ROWS,
    "content_region": EXPECTED_SD_CANDIDATE_ROWS,
    "masked_region": EXPECTED_SD_NONZERO_ROWS,
    "mask_bbox_crop": EXPECTED_SD_NONZERO_ROWS,
    "boundary_region": EXPECTED_SD_NONZERO_ROWS,
    "outside_mask_region": EXPECTED_SD_NONZERO_ROWS,
}


def read_csv_or_empty(path: Path) -> pd.DataFrame:
    if not Path(path).is_file():
        return pd.DataFrame()
    return pd.read_csv(path)


def bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.fillna(False).astype(str).str.strip().str.lower().isin({"true", "1", "yes"})


def read_validation_passed(path: Path) -> bool:
    validation_df = read_csv_or_empty(path)
    if validation_df.empty or "passed" not in validation_df.columns:
        return False
    return bool(bool_series(validation_df["passed"]).all())


def path_exists_from_project_relative(path_value: str) -> bool:
    if pd.isna(path_value) or str(path_value).strip() == "":
        return False
    candidate_path = Path(str(path_value))
    if not candidate_path.is_absolute():
        candidate_path = PROJECT_ROOT / candidate_path
    return candidate_path.exists()


def file_sha256(path: Path) -> str:
    hasher = hashlib.sha256()
    with Path(path).open("rb") as file_obj:
        for block in iter(lambda: file_obj.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()


def path_modified_at_utc(path: Path) -> str:
    if not Path(path).exists():
        return ""
    return datetime.fromtimestamp(Path(path).stat().st_mtime, tz=timezone.utc).isoformat()


def json_safe(value: Any):
    if isinstance(value, Path):
        return rel(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    return value


metric_input_cases_df = read_csv_or_empty(BATCH1_INPUT_CASES_PATH)
stable_diffusion_classical_metrics_df = read_csv_or_empty(BATCH3_METRICS_PATH)
stable_diffusion_classical_metrics_summary_df = read_csv_or_empty(BATCH4_SUMMARY_PATH)
selected_cases_df = read_csv_or_empty(BATCH4_SELECTED_CASES_PATH)
figure_manifest_df = read_csv_or_empty(BATCH4_FIGURE_MANIFEST_PATH)

# Only CSVs with a `passed` column are validation CSVs.
# Some metric/debug CSVs live in validation/ too, e.g. stable_diffusion_classical_metrics_smoke.csv.
validation_dir_csv_paths = sorted(OUTPUT_DIRS["validation"].glob("*.csv"))

def csv_has_passed_column(path: Path) -> bool:
    try:
        return "passed" in pd.read_csv(path, nrows=1).columns
    except Exception:
        return False

previous_validation_paths = [
    path
    for path in validation_dir_csv_paths
    if path.name != BATCH5_FINAL_VALIDATION_PATH.name
    and csv_has_passed_column(path)
]

skipped_non_validation_csv_paths = [
    path
    for path in validation_dir_csv_paths
    if path.name != BATCH5_FINAL_VALIDATION_PATH.name
    and not csv_has_passed_column(path)
]

previous_validation_results = {
    rel(path): read_validation_passed(path)
    for path in previous_validation_paths
}

previous_validation_files_exist = {
    rel(path): path.is_file()
    for path in previous_validation_paths
}

previous_validations_passed = (
    len(previous_validation_results) > 0
    and all(previous_validation_results.values())
)

skipped_non_validation_csvs = {
    rel(path): "skipped because CSV has no passed column"
    for path in skipped_non_validation_csv_paths
}

print("Previous validation files:")
print(json.dumps(previous_validation_results, indent=2))

print("Skipped non-validation CSVs in validation/:")
print(json.dumps(skipped_non_validation_csvs, indent=2))

previous_validation_status = {
    rel(path): read_validation_passed(path)
    for path in previous_validation_paths
}

if STAGE_MANIFEST_PATH.is_file():
    stage_manifest = json.loads(STAGE_MANIFEST_PATH.read_text(encoding="utf-8"))

print(f"Metric input rows: {len(metric_input_cases_df):,}")
print(f"Classical metric rows: {len(stable_diffusion_classical_metrics_df):,}")
print(f"Summary rows: {len(stable_diffusion_classical_metrics_summary_df):,}")
print(f"Selected cases: {len(selected_cases_df):,}")
print(f"Figure manifest rows: {len(figure_manifest_df):,}")
print("Previous validation files:")
print(json.dumps(previous_validation_status, indent=2))

Previous validation files:
{
  "outputs/22_stable_diffusion_classical_metrics/validation/batch0_validation.csv": true,
  "outputs/22_stable_diffusion_classical_metrics/validation/batch1_input_validation.csv": true,
  "outputs/22_stable_diffusion_classical_metrics/validation/batch2_smoke_validation.csv": true,
  "outputs/22_stable_diffusion_classical_metrics/validation/batch4_analysis_validation.csv": true,
  "outputs/22_stable_diffusion_classical_metrics/validation/stable_diffusion_classical_metrics_validation.csv": true
}
Skipped non-validation CSVs in validation/:
{
  "outputs/22_stable_diffusion_classical_metrics/validation/stable_diffusion_classical_metrics_smoke.csv": "skipped because CSV has no passed column"
}
Metric input rows: 945
Classical metric rows: 5,294
Summary rows: 135
Selected cases: 11
Figure manifest rows: 11
Previous validation files:
{
  "outputs/22_stable_diffusion_classical_metrics/validation/batch0_validation.csv": true,
  "outputs/22_stable_diffusion_classical

In [25]:
def artifact_file_metadata(path: Path, artifact_type: str) -> dict:
    path = Path(path)
    exists = path.exists()
    metadata = {
        "exists": bool(exists),
        "size_bytes": 0,
        "file_count": 0,
        "row_count": "",
        "column_count": "",
        "columns": "",
        "sha256": "",
        "modified_at_utc": "",
    }

    if not exists:
        return metadata

    metadata["modified_at_utc"] = path_modified_at_utc(path)

    if path.is_dir():
        files = [item for item in path.rglob("*") if item.is_file()]
        metadata["file_count"] = int(len(files))
        metadata["size_bytes"] = int(sum(item.stat().st_size for item in files))
        return metadata

    metadata["size_bytes"] = int(path.stat().st_size)

    if path.resolve() != Path(ARTIFACT_INDEX_PATH).resolve():
        metadata["sha256"] = file_sha256(path)

    if artifact_type == "csv" and path.is_file():
        try:
            csv_df = pd.read_csv(path)
            metadata["row_count"] = int(len(csv_df))
            metadata["column_count"] = int(len(csv_df.columns))
            metadata["columns"] = " | ".join(map(str, csv_df.columns))
        except Exception as exc:
            metadata["columns"] = f"csv_read_error: {type(exc).__name__}"

    return metadata


def normalize_artifact_spec(spec: dict, default_stage: str) -> dict:
    path = Path(spec["path"])
    artifact_type = spec.get("artifact_type", path.suffix.lstrip(".") or "directory")
    return {
        "artifact_key": spec["artifact_key"],
        "artifact_stage": spec.get("artifact_stage", default_stage),
        "batch": spec.get("batch", default_stage),
        "artifact_type": artifact_type,
        "path": path,
        "required": bool(spec.get("required", True)),
        "notes": spec.get("notes", ""),
    }


def build_artifact_specs() -> list[dict]:
    specs = []
    specs.extend(normalize_artifact_spec(spec, "upstream_input") for spec in REQUIRED_UPSTREAM_ARTIFACTS)
    specs.extend(normalize_artifact_spec(spec, "notebook_output") for spec in PLANNED_OUTPUT_ARTIFACTS)

    extra_specs = [
        {
            "artifact_key": "batch3_validation",
            "batch": "batch3",
            "artifact_type": "csv",
            "path": BATCH3_VALIDATION_PATH,
            "required": True,
            "notes": "Batch 3 full classical metric validation.",
        },
        {
            "artifact_key": "batch4_validation",
            "batch": "batch4",
            "artifact_type": "csv",
            "path": BATCH4_VALIDATION_PATH,
            "required": True,
            "notes": "Batch 4 summary, selected-case, and figure validation.",
        },
        {
            "artifact_key": "batch4_figure_manifest",
            "batch": "batch4",
            "artifact_type": "csv",
            "path": BATCH4_FIGURE_MANIFEST_PATH,
            "required": True,
            "notes": "Manifest for short-name diagnostic PNGs.",
        },
        {
            "artifact_key": "batch4_diagnostic_figure_dir",
            "batch": "batch4",
            "artifact_type": "directory",
            "path": BATCH4_FIGURE_DIR,
            "required": True,
            "notes": "Directory containing Batch 4 diagnostic PNGs.",
        },
    ]

    if "BATCH2_SMOKE_METRICS_PATH" in globals():
        extra_specs.append(
            {
                "artifact_key": "batch2_smoke_metrics",
                "batch": "batch2",
                "artifact_type": "csv",
                "path": BATCH2_SMOKE_METRICS_PATH,
                "required": True,
                "notes": "Batch 2 representative smoke metric output.",
            }
        )

    if "BATCH2_VALIDATION_PATH" in globals():
        extra_specs.append(
            {
                "artifact_key": "batch2_validation",
                "batch": "batch2",
                "artifact_type": "csv",
                "path": BATCH2_VALIDATION_PATH,
                "required": True,
                "notes": "Batch 2 smoke validation.",
            }
        )

    specs.extend(normalize_artifact_spec(spec, "notebook_output") for spec in extra_specs)

    deduped_specs = {}
    for spec in specs:
        deduped_specs[spec["artifact_key"]] = spec
    return list(deduped_specs.values())


def build_artifact_index() -> pd.DataFrame:
    rows = []
    for spec in build_artifact_specs():
        metadata = artifact_file_metadata(spec["path"], spec["artifact_type"])
        rows.append(
            {
                "artifact_key": spec["artifact_key"],
                "artifact_stage": spec["artifact_stage"],
                "batch": spec["batch"],
                "artifact_type": spec["artifact_type"],
                "path": rel(spec["path"]),
                "required": bool(spec["required"]),
                "notes": spec["notes"],
                **metadata,
            }
        )

    return pd.DataFrame(rows).sort_values(
        ["artifact_stage", "batch", "artifact_key"],
        kind="stable",
    ).reset_index(drop=True)


def write_handoff_manifest(final_validation_df: pd.DataFrame, artifact_index_df: pd.DataFrame) -> dict:
    final_validation_passed = bool(bool_series(final_validation_df["passed"]).all())
    required_missing_df = artifact_index_df.loc[
        artifact_index_df["required"].astype(bool) & ~artifact_index_df["exists"].astype(bool)
    ].copy()

    handoff_manifest = {
        "notebook_id": NOTEBOOK_ID,
        "notebook_title": NOTEBOOK_TITLE,
        "completed_at_utc": utc_now_iso(),
        "status": "passed" if final_validation_passed and required_missing_df.empty else "failed",
        "helper": {
            "module": "restoration_eval.metrics_classical",
            "metric_version": metrics_classical.METRIC_VERSION,
            "all_regions": list(metrics_classical.ALL_EVALUATION_REGIONS),
        },
        "counts": {
            "metric_input_rows": int(len(metric_input_cases_df)),
            "classical_metric_rows": int(len(stable_diffusion_classical_metrics_df)),
            "summary_rows": int(len(stable_diffusion_classical_metrics_summary_df)),
            "selected_cases": int(len(selected_cases_df)),
            "figure_manifest_rows": int(len(figure_manifest_df)),
            "validation_rows": int(len(final_validation_df)),
            "validation_passed_rows": int(bool_series(final_validation_df["passed"]).sum()),
            "artifact_index_rows": int(len(artifact_index_df)),
        },
        "expected_counts": {
            "candidate_rows": EXPECTED_SD_CANDIDATE_ROWS,
            "zero_control_rows": EXPECTED_SD_ZERO_CONTROL_ROWS,
            "nonzero_rows": EXPECTED_SD_NONZERO_ROWS,
            "expected_metric_rows": EXPECTED_SD_METRIC_ROWS,
            "expected_region_counts": EXPECTED_BATCH5_REGION_COUNTS,
        },
        "outputs": {
            "metric_input_cases": rel(BATCH1_INPUT_CASES_PATH),
            "classical_metrics": rel(BATCH3_METRICS_PATH),
            "classical_metrics_summary": rel(BATCH4_SUMMARY_PATH),
            "selected_cases": rel(BATCH4_SELECTED_CASES_PATH),
            "figure_manifest": rel(BATCH4_FIGURE_MANIFEST_PATH),
            "diagnostic_figure_dir": rel(BATCH4_FIGURE_DIR),
            "final_validation": rel(BATCH5_FINAL_VALIDATION_PATH),
            "artifact_index": rel(ARTIFACT_INDEX_PATH),
            "stage_manifest": rel(STAGE_MANIFEST_PATH),
            "handoff_manifest": rel(HANDOFF_MANIFEST_PATH),
        },
        "required_missing_artifacts": required_missing_df["artifact_key"].tolist(),
        "previous_validation_status": previous_validation_status,
        "downstream_notes": [
            "Use classical_metrics as the candidate-region metric table.",
            "Use selected_cases plus figure_manifest for visual diagnostics.",
            "Artifact filenames are fixed-name outputs; case details are stored inside CSV/JSON metadata.",
        ],
    }

    HANDOFF_MANIFEST_PATH.write_text(
        json.dumps(json_safe(handoff_manifest), indent=2),
        encoding="utf-8",
    )
    return handoff_manifest

In [26]:
def build_final_validation(artifact_index_df: pd.DataFrame | None = None) -> pd.DataFrame:
    metrics_df = stable_diffusion_classical_metrics_df
    metrics_region_counts = (
        metrics_df["evaluation_region"].value_counts().to_dict()
        if "evaluation_region" in metrics_df.columns
        else {}
    )
    metrics_status_counts = (
        metrics_df["status"].value_counts(dropna=False).to_dict()
        if "status" in metrics_df.columns
        else {}
    )
    metric_version_counts = (
        metrics_df["metric_version"].value_counts(dropna=False).to_dict()
        if "metric_version" in metrics_df.columns
        else {}
    )

    duplicate_candidate_region_rows = (
        int(metrics_df.duplicated(["candidate_id", "evaluation_region"], keep=False).sum())
        if {"candidate_id", "evaluation_region"}.issubset(metrics_df.columns)
        else len(metrics_df)
    )

    zero_control_rows = (
        int(metric_input_cases_df["mask_type"].astype(str).eq(ZERO_CONTROL_MASK_TYPE).sum())
        if "mask_type" in metric_input_cases_df.columns
        else 0
    )
    nonzero_rows = int(len(metric_input_cases_df) - zero_control_rows)

    figure_paths_exist = []
    if not figure_manifest_df.empty and "figure_path" in figure_manifest_df.columns:
        figure_paths_exist = [
            path_exists_from_project_relative(path_value)
            for path_value in figure_manifest_df["figure_path"].astype(str)
        ]

    figure_filename_short = True
    if not figure_manifest_df.empty and "figure_filename_length" in figure_manifest_df.columns:
        figure_filename_short = bool((figure_manifest_df["figure_filename_length"].astype(int) <= 20).all())
    elif not figure_manifest_df.empty and "figure_filename" in figure_manifest_df.columns:
        figure_filename_short = bool((figure_manifest_df["figure_filename"].astype(str).str.len() <= 20).all())

    required_missing = []
    if artifact_index_df is not None and not artifact_index_df.empty:
        required_missing = artifact_index_df.loc[
            artifact_index_df["required"].astype(bool) & ~artifact_index_df["exists"].astype(bool),
            "artifact_key",
        ].tolist()

    validation_rows = [
        validation_row(
            "batch5_previous_validation_files_present",
            len(previous_validation_status),
            ">= 4 validation files before final validation",
            len(previous_validation_status) >= 4,
            "Expected Batch 0/2/3/4 validation outputs before final validation.",
        ),
        validation_row(
            "batch5_previous_validations_passed",
            previous_validation_results,
            "all true",
            previous_validations_passed,
            "One or more previous validation CSVs with a passed column did not pass.",
        ),
        validation_row(
            "batch5_metric_input_exists",
            rel(BATCH1_INPUT_CASES_PATH),
            "file exists",
            BATCH1_INPUT_CASES_PATH.is_file(),
            "Metric input cases CSV is missing.",
        ),
        validation_row(
            "batch5_metric_input_rows",
            len(metric_input_cases_df),
            EXPECTED_SD_CANDIDATE_ROWS,
            len(metric_input_cases_df) == EXPECTED_SD_CANDIDATE_ROWS,
            "Metric input row count is not 945.",
        ),
        validation_row(
            "batch5_zero_control_rows",
            zero_control_rows,
            EXPECTED_SD_ZERO_CONTROL_ROWS,
            zero_control_rows == EXPECTED_SD_ZERO_CONTROL_ROWS,
            "Zero-control input count is not 94.",
        ),
        validation_row(
            "batch5_nonzero_rows",
            nonzero_rows,
            EXPECTED_SD_NONZERO_ROWS,
            nonzero_rows == EXPECTED_SD_NONZERO_ROWS,
            "Non-zero input count is not 851.",
        ),
        validation_row(
            "batch5_classical_metrics_exists",
            rel(BATCH3_METRICS_PATH),
            "file exists",
            BATCH3_METRICS_PATH.is_file(),
            "Full classical metrics CSV is missing.",
        ),
        validation_row(
            "batch5_classical_metric_rows",
            len(metrics_df),
            EXPECTED_SD_METRIC_ROWS,
            len(metrics_df) == EXPECTED_SD_METRIC_ROWS,
            "Full classical metric row count is not 5294.",
        ),
        validation_row(
            "batch5_classical_metric_status_ok",
            metrics_status_counts,
            {"ok": EXPECTED_SD_METRIC_ROWS},
            metrics_status_counts == {"ok": EXPECTED_SD_METRIC_ROWS},
            "Classical metric table contains non-ok rows.",
        ),
        validation_row(
            "batch5_classical_metric_region_counts",
            metrics_region_counts,
            EXPECTED_BATCH5_REGION_COUNTS,
            metrics_region_counts == EXPECTED_BATCH5_REGION_COUNTS,
            "Classical metric region counts do not match the expected Notebook 22 contract.",
        ),
        validation_row(
            "batch5_classical_metric_candidate_region_unique",
            duplicate_candidate_region_rows,
            0,
            duplicate_candidate_region_rows == 0,
            "candidate_id + evaluation_region is not unique in the full metric table.",
        ),
        validation_row(
            "batch5_classical_metric_version",
            metric_version_counts,
            {metrics_classical.METRIC_VERSION: EXPECTED_SD_METRIC_ROWS},
            metric_version_counts == {metrics_classical.METRIC_VERSION: EXPECTED_SD_METRIC_ROWS},
            "Classical metric table does not consistently use the expected helper version.",
        ),
        validation_row(
            "batch5_summary_exists",
            rel(BATCH4_SUMMARY_PATH),
            "file exists",
            BATCH4_SUMMARY_PATH.is_file(),
            "Batch 4 summary CSV is missing.",
        ),
        validation_row(
            "batch5_summary_not_empty",
            len(stable_diffusion_classical_metrics_summary_df),
            "> 0",
            len(stable_diffusion_classical_metrics_summary_df) > 0,
            "Batch 4 summary table is empty.",
        ),
        validation_row(
            "batch5_selected_cases_exists",
            rel(BATCH4_SELECTED_CASES_PATH),
            "file exists",
            BATCH4_SELECTED_CASES_PATH.is_file(),
            "Batch 4 selected-cases CSV is missing.",
        ),
        validation_row(
            "batch5_selected_cases_not_empty",
            len(selected_cases_df),
            "> 0",
            len(selected_cases_df) > 0,
            "Batch 4 selected-cases table is empty.",
        ),
        validation_row(
            "batch5_figure_manifest_exists",
            rel(BATCH4_FIGURE_MANIFEST_PATH),
            "file exists",
            BATCH4_FIGURE_MANIFEST_PATH.is_file(),
            "Batch 4 figure manifest is missing.",
        ),
        validation_row(
            "batch5_figure_files_exist",
            int(sum(figure_paths_exist)),
            len(figure_paths_exist),
            bool(figure_paths_exist) and all(figure_paths_exist),
            "One or more Batch 4 diagnostic figures are missing.",
        ),
        validation_row(
            "batch5_short_figure_filenames",
            figure_filename_short,
            True,
            figure_filename_short,
            "One or more Batch 4 figure filenames are longer than the fixed short-name policy.",
        ),
        validation_row(
            "batch5_artifact_index_written",
            rel(ARTIFACT_INDEX_PATH),
            "file exists",
            ARTIFACT_INDEX_PATH.is_file(),
            "Artifact index was not written.",
        ),
        validation_row(
            "batch5_handoff_manifest_written",
            rel(HANDOFF_MANIFEST_PATH),
            "file exists",
            HANDOFF_MANIFEST_PATH.is_file(),
            "Handoff manifest was not written.",
        ),
        validation_row(
            "batch5_artifact_index_required_artifacts_exist",
            required_missing,
            [],
            artifact_index_df is not None and len(required_missing) == 0,
            "One or more required artifacts are missing from disk.",
        ),
    ]

    return pd.DataFrame(validation_rows)


# Write once so the final validation artifact exists before the final artifact-index pass.
preliminary_artifact_index_df = build_artifact_index()
preliminary_validation_df = build_final_validation(preliminary_artifact_index_df)
BATCH5_FINAL_VALIDATION_PATH.write_text(preliminary_validation_df.to_csv(index=False), encoding="utf-8")

# Build and write the closeout artifacts, then refresh validation/index/handoff in final order.
artifact_index_df = build_artifact_index()
artifact_index_df.to_csv(ARTIFACT_INDEX_PATH, index=False)
handoff_manifest = write_handoff_manifest(preliminary_validation_df, artifact_index_df)

artifact_index_df = build_artifact_index()
final_validation_df = build_final_validation(artifact_index_df)
final_validation_df.to_csv(BATCH5_FINAL_VALIDATION_PATH, index=False)

artifact_index_df = build_artifact_index()
artifact_index_df.to_csv(ARTIFACT_INDEX_PATH, index=False)
handoff_manifest = write_handoff_manifest(final_validation_df, artifact_index_df)

artifact_index_df = build_artifact_index()
artifact_index_df.to_csv(ARTIFACT_INDEX_PATH, index=False)

stage_manifest["batch5"] = {
    "completed_at_utc": utc_now_iso(),
    "status": "passed" if bool_series(final_validation_df["passed"]).all() else "failed",
    "final_validation_rows": int(len(final_validation_df)),
    "final_validation_passed_rows": int(bool_series(final_validation_df["passed"]).sum()),
    "artifact_index_rows": int(len(artifact_index_df)),
    "required_artifacts_missing": artifact_index_df.loc[
        artifact_index_df["required"].astype(bool) & ~artifact_index_df["exists"].astype(bool),
        "artifact_key",
    ].tolist(),
    "outputs": {
        "final_validation": rel(BATCH5_FINAL_VALIDATION_PATH),
        "artifact_index": rel(ARTIFACT_INDEX_PATH),
        "handoff_manifest": rel(HANDOFF_MANIFEST_PATH),
        "stage_manifest": rel(STAGE_MANIFEST_PATH),
    },
}

stage_manifest["completed_at_utc"] = utc_now_iso()
stage_manifest["status"] = stage_manifest["batch5"]["status"]
stage_manifest["artifact_index"] = rel(ARTIFACT_INDEX_PATH)
stage_manifest["handoff_manifest"] = rel(HANDOFF_MANIFEST_PATH)
stage_manifest["final_validation"] = rel(BATCH5_FINAL_VALIDATION_PATH)

STAGE_MANIFEST_PATH.write_text(json.dumps(json_safe(stage_manifest), indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH5_FINAL_VALIDATION_PATH)}")
print(f"Saved: {rel(ARTIFACT_INDEX_PATH)}")
print(f"Saved: {rel(HANDOFF_MANIFEST_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 5 checks passed: {int(bool_series(final_validation_df['passed']).sum())} / {len(final_validation_df)}")

display(final_validation_df)
display(artifact_index_df)

if not bool_series(final_validation_df["passed"]).all():
    display(final_validation_df.loc[~bool_series(final_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 5 final validation failed. Fix closeout artifacts before moving downstream.")

Saved: outputs/22_stable_diffusion_classical_metrics/validation/stable_diffusion_classical_metrics_final_validation.csv
Saved: outputs/22_stable_diffusion_classical_metrics/manifests/stable_diffusion_classical_metrics_artifact_index.csv
Saved: outputs/22_stable_diffusion_classical_metrics/manifests/stable_diffusion_classical_metrics_handoff_manifest.json
Updated: outputs/22_stable_diffusion_classical_metrics/manifests/stable_diffusion_classical_metrics_stage_manifest.json
Batch 5 checks passed: 22 / 22


,check_name,observed,expected,passed,failure_message
0,batch5_previous_validation_files_present,5,>= 4 validation files before final validation,True,
1,batch5_previous_validations_passed,{'outputs/22_stable_diffusion_classical_metric...,all true,True,
2,batch5_metric_input_exists,outputs/22_stable_diffusion_classical_metrics/...,file exists,True,
3,batch5_metric_input_rows,945,945,True,
4,batch5_zero_control_rows,94,94,True,
5,batch5_nonzero_rows,851,851,True,
6,batch5_classical_metrics_exists,outputs/22_stable_diffusion_classical_metrics/...,file exists,True,
7,batch5_classical_metric_rows,5294,5294,True,
8,batch5_classical_metric_status_ok,{'ok': 5294},{'ok': 5294},True,
9,batch5_classical_metric_region_counts,"{'full_image': 945, 'content_region': 945, 'ma...","{'full_image': 945, 'content_region': 945, 'ma...",True,


,artifact_key,artifact_stage,batch,artifact_type,path,required,notes,exists,size_bytes,file_count,row_count,column_count,columns,sha256,modified_at_utc
0,batch0_project_inventory_snapshot,notebook_output,batch0,csv,outputs/22_stable_diffusion_classical_metrics/...,True,Inventory snapshot captured at notebook start.,True,1693039,0,4123,17,relative_path | file_name | parent_dir | exten...,018704067de71db6d41b0ea12492dd9015ac5f8174fa5e...,2026-08-07T17:34:24.025045+00:00
1,batch0_validation,notebook_output,batch0,csv,outputs/22_stable_diffusion_classical_metrics/...,True,Batch 0 validation checks.,True,1174,0,9,5,check_name | observed | expected | passed | fa...,7299881208643d539b08caf0e0e82c16c7ba907f881a90...,2026-08-07T17:39:24.068857+00:00
2,stage_manifest,notebook_output,batch0_to_final,json,outputs/22_stable_diffusion_classical_metrics/...,True,Notebook-wide stage manifest updated during th...,True,14058,0,,,,b2063f2f119cef3501ba9d15a66d40b822dafba4eab5f1...,2026-08-07T18:48:19.133408+00:00
3,metric_input_cases,notebook_output,batch1,csv,outputs/22_stable_diffusion_classical_metrics/...,True,Standardized candidate-level metric input table.,True,3535074,0,945,278,metric_case_id | dataset_name | case_id | rest...,d424c5eeda5f02f2a1a98ba65c51529a7e35814b7744a0...,2026-08-07T17:50:24.120493+00:00
4,batch2_smoke_metrics,notebook_output,batch2,csv,outputs/22_stable_diffusion_classical_metrics/...,True,Batch 2 representative smoke metric output.,True,31707,0,28,58,dataset_name | case_id | painting_id | categor...,7af68658294bf11dee85726f686da977c5a2b42a3bfc44...,2026-08-07T18:00:42.850642+00:00
5,batch2_validation,notebook_output,batch2,csv,outputs/22_stable_diffusion_classical_metrics/...,True,Batch 2 smoke validation.,True,749,0,12,5,check_name | observed | expected | passed | fa...,a65a64b492422900b3b58bff642a4b0fb1d20465fda497...,2026-08-07T18:00:42.854773+00:00
6,batch3_validation,notebook_output,batch3,csv,outputs/22_stable_diffusion_classical_metrics/...,True,Batch 3 full classical metric validation.,True,1672,0,18,5,check_name | observed | expected | passed | fa...,4eb7b81b5fa033b92f59c65465b6167ec3da1ac565760d...,2026-08-07T18:39:55.629318+00:00
7,classical_metrics,notebook_output,batch3,csv,outputs/22_stable_diffusion_classical_metrics/...,True,Main candidate-level classical metrics table.,True,6288950,0,5294,58,dataset_name | case_id | painting_id | categor...,afe6e2484bb5b4a0838611dfb29dcc1d094feb9a1a0d2e...,2026-08-07T18:39:55.629318+00:00
8,batch4_diagnostic_figure_dir,notebook_output,batch4,directory,outputs/22_stable_diffusion_classical_metrics/...,True,Directory containing Batch 4 diagnostic PNGs.,True,8766620,11,,,,,2026-08-07T18:40:19.834927+00:00
9,batch4_figure_manifest,notebook_output,batch4,csv,outputs/22_stable_diffusion_classical_metrics/...,True,Manifest for short-name diagnostic PNGs.,True,5764,0,11,19,selected_case_index | figure_id | figure_filen...,385cac9104a432346e123ae6f8b2505d63e998ad762787...,2026-08-07T18:40:20.278884+00:00
